# KPI-Aware Contrastive with Uncertainty Weighting — TelecomTS Evaluation

This notebook implements the **KPI-Aware Contrastive (KAC)** method with **uncertainty-weighted KPI contrastive loss** for multimodal anomaly detection on the **balanced TelecomTS** dataset.

## Method Overview

**Architecture:**
- **Text encoder:** DistilBERT (last 2 layers unfrozen) encodes textual KPI descriptions
- **Time-series encoder:** Pre-computed Chronos residuals (actual − forecast) capture temporal anomaly signals
- **KPI query cross-attention:** K learnable query embeddings cross-attend to text tokens → per-KPI text representations
- **Per-KPI residual projector:** Maps each KPI's residual channels to a shared space for contrastive alignment
- **Gated fusion:** Text-gated residual modulation + Conv1D + CLS-pooling attention
- **Classification head:** LayerNorm → MLP → binary logit

**Loss:** `L = L_BCE + α · L_SupCon + β · L_KPI-Contrastive(uncertainty-weighted)`

The **uncertainty-weighted** KPI contrastive loss uses Chronos prediction-interval widths to down-weight KPI channels where the forecast is uncertain and up-weight channels where the model is confident. This focuses alignment on the most informative KPIs.

## Dataset

| Split | Samples | Anomaly (pos) | Normal (neg) | Balance |
|-------|---------|---------------|--------------|--------|
| Train | 1,580 | 790 | 790 | 50/50 |
| Val | 396 | 198 | 198 | 50/50 |
| Test | 494 | 247 | 247 | 50/50 |

Each sample: 128 time steps × 16 KPIs (raw), 108 time steps × 80 features (Chronos residuals: 5 features per KPI × 16 KPIs), and a text description.

**Update:** The neural network consumes z-normalized residuals, while uncertainty-weighted KPI contrastive learning uses a separate raw Chronos prediction-interval-width tensor extracted before normalization.


## 1. Imports and Configuration

In [ ]:
import os, re, copy, math, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from sklearn.metrics import (
    precision_recall_fscore_support, accuracy_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
)
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

## 2. Load TelecomTS Dataset and Chronos Residuals

In [ ]:
DATA_DIR   = Path("data")
CACHE_DIR  = DATA_DIR / "features_cache"

train_npz = np.load(DATA_DIR / "TelecomTS_train.npz", allow_pickle=True)
val_npz   = np.load(DATA_DIR / "TelecomTS_val.npz",   allow_pickle=True)
test_npz  = np.load(DATA_DIR / "TelecomTS_test.npz",  allow_pickle=True)

X_train_raw = train_npz["X"]
X_val_raw   = val_npz["X"]
X_test_raw  = test_npz["X"]

y_train = np.array(train_npz["y"], dtype=np.int64)
y_val   = np.array(val_npz["y"],   dtype=np.int64)
y_test  = np.array(test_npz["y"],  dtype=np.int64)

texts_train = list(train_npz["descriptions"])
texts_val   = list(val_npz["descriptions"])
texts_test  = list(test_npz["descriptions"])

kpi_names = (
    [str(x) for x in train_npz["feature_cols"]]
    if "feature_cols" in train_npz.files
    else [f"KPI_{i}" for i in range(X_train_raw.shape[-1])]
)

# Load pre-computed Chronos residuals.
# Keep an unnormalized copy so the uncertainty loss can use the raw
# Chronos prediction-interval width (q95 - q05). The neural network
# still receives z-normalized residual features for stable training.
R_train_raw = np.load(CACHE_DIR / "residuals_train.npy").astype(np.float32)
R_val_raw   = np.load(CACHE_DIR / "residuals_val.npy").astype(np.float32)
R_test_raw  = np.load(CACHE_DIR / "residuals_test.npy").astype(np.float32)

N_KPIS_RAW = len(kpi_names)
FEATS_PER_KPI_RAW = R_train_raw.shape[-1] // N_KPIS_RAW
WIDTH_IDX = 3
assert R_train_raw.shape[-1] == N_KPIS_RAW * FEATS_PER_KPI_RAW, (
    f"Residual feature dimension {R_train_raw.shape[-1]} is not divisible "
    f"by number of KPIs {N_KPIS_RAW}"
)
assert WIDTH_IDX < FEATS_PER_KPI_RAW, (
    f"width_idx={WIDTH_IDX} is invalid for feats_per_kpi={FEATS_PER_KPI_RAW}"
)

_width_cols = [k * FEATS_PER_KPI_RAW + WIDTH_IDX for k in range(N_KPIS_RAW)]
_neg_width_frac = float((R_train_raw[:, :, _width_cols] < 0).mean())
if _neg_width_frac > 0.01:
    print(
        f"WARNING: {_neg_width_frac:.2%} of training interval widths are negative. "
        "Prediction-interval widths should be nonnegative; check that residual cache "
        "contains raw q95-q05 widths rather than already-normalized values."
    )
W_train_raw = np.abs(R_train_raw[:, :, _width_cols]).astype(np.float32)
W_val_raw   = np.abs(R_val_raw[:,   :, _width_cols]).astype(np.float32)
W_test_raw  = np.abs(R_test_raw[:,  :, _width_cols]).astype(np.float32)

# Z-normalize residual features using training-set statistics only.
# These normalized tensors are the numerical inputs to KAC.
r_mean = R_train_raw.mean(axis=(0, 1), keepdims=True)
r_std  = R_train_raw.std(axis=(0, 1), keepdims=True) + 1e-8
R_train = ((R_train_raw - r_mean) / r_std).astype(np.float32)
R_val   = ((R_val_raw   - r_mean) / r_std).astype(np.float32)
R_test  = ((R_test_raw  - r_mean) / r_std).astype(np.float32)

print(f"Raw interval-width tensors: train {W_train_raw.shape}, val {W_val_raw.shape}, test {W_test_raw.shape}")
print(f"Residual features normalized for model input; raw widths kept for uncertainty weighting.")

print("─" * 50)
print("Dataset shapes")
print("─" * 50)
print(f"X_train : {X_train_raw.shape}  |  R_train : {R_train.shape}")
print(f"X_val   : {X_val_raw.shape}    |  R_val   : {R_val.shape}")
print(f"X_test  : {X_test_raw.shape}    |  R_test  : {R_test.shape}")
print(f"\nTrain positives: {y_train.sum()} / {len(y_train)}")
print(f"Val   positives: {y_val.sum()} / {len(y_val)}")
print(f"Test  positives: {y_test.sum()} / {len(y_test)}")
print(f"\nKPIs ({len(kpi_names)}): {kpi_names}")


## 3. Anomaly-Word Masking (Leakage Prevention)

We mask words that directly reveal the anomaly label (e.g. "anomaly", "degraded", "failure") to prevent text-to-label information leakage.

In [ ]:
ANOMALY_WORDS = [
    r'\banomaly\b', r'\banomalous\b', r'\babnormal\b', r'\bdegraded\b',
    r'\bdegradation\b', r'\bfault\b', r'\bfailure\b', r'\bcritical\b',
    r'\bsevere\b', r'\balarm\b', r'\bdrop\b', r'\bdropped\b',
    r'\bdeteriorate\b', r'\bdeterioration\b', r'\bexcessively\b',
    r'\bplummet\b', r'\bplummeted\b', r'\bsurge\b', r'\bspiked?\b',
    r'\bcollapse\b', r'\bcongestion\b', r'\bcongested\b',
    r'\bbottleneck\b', r'\boverloaded?\b', r'\bexhausted\b',
    r'\binterferen\w*\b', r'\bhigh.?error\b', r'\blow.?quality\b',
]
MASK_PATTERN = re.compile("|".join(ANOMALY_WORDS), flags=re.IGNORECASE)

def mask_text(text):
    return MASK_PATTERN.sub("[MASK]", str(text))

texts_train_masked = [mask_text(t) for t in texts_train]
texts_val_masked   = [mask_text(t) for t in texts_val]
texts_test_masked  = [mask_text(t) for t in texts_test]

print("Original (first 200 chars):")
print(f"  {texts_train[0][:200]}")
print("\nMasked (first 200 chars):")
print(f"  {texts_train_masked[0][:200]}")

## 4. Tokenize Text with DistilBERT

In [ ]:
TEXT_MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME, local_files_only=True)

train_tokens = tokenizer(
    texts_train_masked, padding="max_length", truncation=True,
    max_length=MAX_LEN, return_tensors="pt",
)
val_tokens = tokenizer(
    texts_val_masked, padding="max_length", truncation=True,
    max_length=MAX_LEN, return_tensors="pt",
)
test_tokens = tokenizer(
    texts_test_masked, padding="max_length", truncation=True,
    max_length=MAX_LEN, return_tensors="pt",
)

full_lengths = [len(tokenizer.encode(t, add_special_tokens=True)) for t in texts_train_masked]
print(f"Full text token lengths — mean: {np.mean(full_lengths):.0f}, "
      f"median: {np.median(full_lengths):.0f}, max: {np.max(full_lengths)}")
print(f"MAX_LEN={MAX_LEN} captures {100*np.mean([l <= MAX_LEN for l in full_lengths]):.1f}% "
      f"of train samples fully")
print(f"\ntrain input_ids : {train_tokens['input_ids'].shape}")
print(f"val   input_ids : {val_tokens['input_ids'].shape}")
print(f"test  input_ids : {test_tokens['input_ids'].shape}")

## 5. DistilBERT Encoder (Selective Unfreezing)

In [ ]:
import threading
from transformers import DistilBertConfig, DistilBertModel
from safetensors.torch import load_file as _load_safetensors
from huggingface_hub import try_to_load_from_cache

# ── Fix leaked accelerate init_empty_weights() monkey-patch ──────
# A prior AutoModel.from_pretrained() call may have permanently patched
# nn.Module.register_parameter / register_buffer to create meta tensors.
if torch.nn.Linear(1, 1).weight.device.type == "meta":
    _Param, _Tensor = torch.nn.Parameter, torch.Tensor
    def _rp(self, name, param):
        if param is not None and not isinstance(param, _Param):
            raise TypeError(f"expected Parameter, got {type(param)}")
        self._parameters[name] = param
    def _rb(self, name, tensor, persistent=True):
        if tensor is not None and not isinstance(tensor, _Tensor):
            raise TypeError(f"expected Tensor, got {type(tensor)}")
        self._buffers[name] = tensor
        if not persistent:
            self._non_persistent_buffers_set.add(name)
    torch.nn.Module.register_parameter = _rp
    torch.nn.Module.register_buffer = _rb
    assert torch.nn.Linear(1, 1).weight.device.type == "cpu"
    print("Restored nn.Module.register_parameter/buffer (was meta-patched by accelerate)")

UNFREEZE_LAST_N = 2
_model_load_lock = threading.Lock()
_base_config = None
_base_sd = None

def load_text_encoder(unfreeze_last_n=UNFREEZE_LAST_N):
    global _base_config, _base_sd
    with _model_load_lock:
        if _base_sd is None:
            _base_config = DistilBertConfig.from_pretrained(
                TEXT_MODEL_NAME, local_files_only=True,
            )
            wpath = try_to_load_from_cache(TEXT_MODEL_NAME, "model.safetensors")
            raw_sd = _load_safetensors(wpath, device="cpu")
            prefix = "distilbert."
            _base_sd = {
                k[len(prefix):]: v for k, v in raw_sd.items()
                if k.startswith(prefix)
            }
    enc = DistilBertModel(_base_config)
    enc.load_state_dict(_base_sd)
    enc.requires_grad_(False)
    for layer in enc.transformer.layer[-unfreeze_last_n:]:
        for param in layer.parameters():
            param.requires_grad = True
    n_total = sum(p.numel() for p in enc.parameters())
    n_train = sum(p.numel() for p in enc.parameters() if p.requires_grad)
    print(f"DistilBERT: {n_train:,} / {n_total:,} params trainable "
          f"({100 * n_train / n_total:.1f}%)")
    return enc

text_encoder = load_text_encoder()

## 6. Loss Functions

**Three loss terms:**
1. **L_BCE**: Binary cross-entropy for anomaly classification
2. **L_SupCon**: Supervised contrastive loss — clusters same-class fused representations (Khosla et al., NeurIPS 2020)
3. **L_KPI (uncertainty-weighted)**: Per-KPI CLIP-style InfoNCE, with per-KPI weights derived from Chronos prediction-interval widths — confident KPIs contribute more

In [ ]:
class SupConLoss(nn.Module):
    """Supervised Contrastive Loss (Khosla et al., NeurIPS 2020)."""

    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        device = features.device
        B = features.shape[0]
        if B <= 1:
            return torch.tensor(0.0, device=device, requires_grad=True)

        sim = features @ features.T / self.temperature

        lab = labels.view(-1, 1)
        mask_pos = (lab == lab.T).float()
        mask_pos.fill_diagonal_(0)

        n_pos = mask_pos.sum(dim=1)
        valid = n_pos > 0
        if valid.sum() == 0:
            return torch.tensor(0.0, device=device, requires_grad=True)

        mask_self = torch.eye(B, dtype=torch.bool, device=device)
        sim = sim.masked_fill(mask_self, -1e9)

        log_prob = sim - torch.logsumexp(sim, dim=1, keepdim=True)
        mean_log_prob = (mask_pos * log_prob).sum(dim=1) / n_pos.clamp(min=1)

        return -mean_log_prob[valid].mean()


def kpi_aware_contrastive_loss(z_text_kpi, z_resid_kpi, temperature=0.07):
    """Per-KPI CLIP-style InfoNCE (base version, used internally)."""
    B, K, _ = z_text_kpi.shape
    labels = torch.arange(B, device=z_text_kpi.device)
    total = 0.0
    for k in range(K):
        logits = z_text_kpi[:, k] @ z_resid_kpi[:, k].T / temperature
        total += (F.cross_entropy(logits, labels)
                  + F.cross_entropy(logits.T, labels)) / 2
    return total / K


def kpi_uncertainty_weights(width_raw, eps=1e-6):
    """Per-(batch, KPI) weight from raw Chronos prediction-interval widths.

    width_raw has shape [B, T_r, K] and is extracted before residual
    z-normalization. Narrow intervals imply higher forecast confidence and
    therefore a larger contrastive weight.
    """
    if width_raw.dim() != 3:
        raise ValueError(f"width_raw must have shape [B, T_r, K], got {tuple(width_raw.shape)}")
    mean_width = width_raw.abs().mean(dim=1)  # [B, K]
    W = 1.0 / (mean_width + eps)
    W = W / (W.mean(dim=1, keepdim=True) + eps)
    return W


def kpi_uncertainty_weighted_contrastive_loss(
    z_text_kpi, z_resid_kpi, width_raw, temperature=0.07
):
    """Uncertainty-weighted per-KPI InfoNCE.

    z_text_kpi, z_resid_kpi: [B, K, p]
    width_raw: raw Chronos interval widths [B, T_r, K], not z-normalized.
    """
    B, K, _ = z_text_kpi.shape
    device = z_text_kpi.device
    if width_raw.shape[0] != B or width_raw.shape[2] != K:
        raise ValueError(
            f"width_raw shape {tuple(width_raw.shape)} is incompatible with "
            f"KPI embeddings shape {tuple(z_text_kpi.shape)}"
        )

    labels = torch.arange(B, device=device)
    Wmat = kpi_uncertainty_weights(width_raw.to(device))  # [B, K]

    w_list, l_list = [], []
    for k in range(K):
        logits = z_text_kpi[:, k] @ z_resid_kpi[:, k].T / temperature
        loss_k = (
            F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)
        ) / 2
        w_k = Wmat[:, k].mean()
        w_list.append(w_k)
        l_list.append(loss_k)

    w = torch.stack(w_list)
    ell = torch.stack(l_list)
    return (w * ell).sum() / w.sum().clamp(min=1e-8)


print("Loss functions defined.")


## 7. Model Architecture

In [ ]:
N_KPIS = len(kpi_names)
FEATS_PER_KPI = R_train.shape[-1] // N_KPIS
print(f"N_KPIS = {N_KPIS}, FEATS_PER_KPI = {FEATS_PER_KPI}")


class KPIAwareContrastiveModel(nn.Module):
    def __init__(
        self, text_encoder, resid_dim, text_dim, d_model, proj_dim,
        n_kpis, feats_per_kpi, num_heads=4, dropout=0.1,
    ):
        super().__init__()
        self.text_encoder = text_encoder
        self.n_kpis = n_kpis
        self.feats_per_kpi = feats_per_kpi

        self.text_proj  = nn.Linear(text_dim, d_model)
        self.resid_proj = nn.Linear(resid_dim, d_model)

        self.kpi_queries = nn.Parameter(torch.randn(n_kpis, d_model) * 0.02)
        self.kpi_text_attn = nn.MultiheadAttention(
            d_model, num_heads, dropout=dropout, batch_first=True,
        )
        self.resid_kpi_proj = nn.Linear(feats_per_kpi, d_model)

        self.kpi_text_cp  = nn.Linear(d_model, proj_dim)
        self.kpi_resid_cp = nn.Linear(d_model, proj_dim)

        self.supcon_proj = nn.Sequential(
            nn.Linear(d_model, d_model), nn.ReLU(), nn.Linear(d_model, proj_dim),
        )

        self.gate = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, d_model),
        )
        self.conv = nn.Conv1d(d_model, d_model, 3, padding=1)
        self.drop = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pool_attn = nn.MultiheadAttention(
            d_model, num_heads, dropout=dropout, batch_first=True,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model, 1),
        )

    def forward(self, input_ids, attention_mask, x_resid):
        text_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        t = self.text_proj(text_out.last_hidden_state)
        r = self.resid_proj(x_resid)
        B = t.shape[0]

        kpi_q = self.kpi_queries.unsqueeze(0).expand(B, -1, -1)
        key_pad = (attention_mask == 0)
        kpi_text, kpi_attn_w = self.kpi_text_attn(
            query=kpi_q, key=t, value=t,
            key_padding_mask=key_pad, need_weights=True,
        )

        kpi_resid_list = []
        for k in range(self.n_kpis):
            s, e = k * self.feats_per_kpi, (k + 1) * self.feats_per_kpi
            r_k = self.resid_kpi_proj(x_resid[:, :, s:e]).mean(dim=1)
            kpi_resid_list.append(r_k)
        kpi_resid = torch.stack(kpi_resid_list, dim=1)

        z_t_kpi = F.normalize(self.kpi_text_cp(kpi_text), dim=-1)
        z_r_kpi = F.normalize(self.kpi_resid_cp(kpi_resid), dim=-1)

        mask_f = attention_mask.unsqueeze(-1).float()
        t_pool = (t * mask_f).sum(1) / mask_f.sum(1).clamp(min=1)
        gate = torch.sigmoid(self.gate(t_pool)).unsqueeze(1)
        r_g = r * gate

        x = torch.cat([t, r_g], dim=1)
        x = self.norm(x + self.drop(self.conv(x.transpose(1, 2)).transpose(1, 2)))
        cls = self.cls_token.expand(B, -1, -1)
        pooled, _ = self.pool_attn(query=cls, key=x, value=x)
        pooled = pooled.squeeze(1)

        z_sc = F.normalize(self.supcon_proj(pooled), dim=-1)
        logits = self.head(pooled).squeeze(-1)

        return logits, z_sc, z_t_kpi, z_r_kpi, kpi_attn_w


print("Model architecture defined.")

## 8. Dataset and DataLoaders

In [ ]:
class ContrastiveDataset(Dataset):
    def __init__(self, input_ids, attention_mask, residuals, labels, interval_widths=None):
        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.residuals = torch.tensor(residuals, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32)
        self.interval_widths = (
            torch.tensor(interval_widths, dtype=torch.float32)
            if interval_widths is not None else None
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "resid": self.residuals[idx],
            "label": self.labels[idx],
        }
        if self.interval_widths is not None:
            item["width_raw"] = self.interval_widths[idx]
        return item


BATCH_SIZE = 16

def make_loaders(batch_size=BATCH_SIZE, seed=SEED):
    train_ds = ContrastiveDataset(
        train_tokens["input_ids"], train_tokens["attention_mask"],
        R_train, y_train, interval_widths=W_train_raw,
    )
    val_ds = ContrastiveDataset(
        val_tokens["input_ids"], val_tokens["attention_mask"],
        R_val, y_val, interval_widths=W_val_raw,
    )
    test_ds = ContrastiveDataset(
        test_tokens["input_ids"], test_tokens["attention_mask"],
        R_test, y_test, interval_widths=W_test_raw,
    )
    g = torch.Generator()
    g.manual_seed(seed)
    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True, generator=g),
        DataLoader(val_ds,   batch_size=batch_size, shuffle=False),
        DataLoader(test_ds,  batch_size=batch_size, shuffle=False),
    )


print(f"Batch size: {BATCH_SIZE}")
print(f"Train batches: {math.ceil(len(y_train) / BATCH_SIZE)}")
print(f"Val   batches: {math.ceil(len(y_val) / BATCH_SIZE)}")
print(f"Test  batches: {math.ceil(len(y_test) / BATCH_SIZE)}")


## 9. Evaluation and Training Functions

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion_bce, device=DEVICE, threshold=0.5):
    model.eval()
    all_logits, all_labels = [], []
    total_loss, total_n = 0.0, 0
    for batch in loader:
        ids   = batch["input_ids"].to(device)
        mask  = batch["attention_mask"].to(device)
        resid = batch["resid"].to(device)
        y     = batch["label"].to(device)
        logits = model(ids, mask, resid)[0]
        loss = criterion_bce(logits, y)
        total_loss += loss.item() * y.size(0)
        total_n += y.size(0)
        all_logits.append(logits.cpu())
        all_labels.append(y.cpu())
    al = torch.cat(all_logits).numpy()
    yl = torch.cat(all_labels).numpy()
    probs = 1.0 / (1.0 + np.exp(-al))
    preds = (probs >= threshold).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(yl, preds, average="binary", zero_division=0)
    acc = accuracy_score(yl, preds)
    try:    auroc = roc_auc_score(yl, probs)
    except: auroc = float("nan")
    try:    ap = average_precision_score(yl, probs)
    except: ap = float("nan")
    return {"loss": total_loss/max(total_n,1), "precision": p, "recall": r,
            "f1": f1, "acc": acc, "auroc": auroc, "ap": ap,
            "probs": probs, "preds": preds, "labels": yl}


def train_kac_uncertainty(
    model, train_loader, val_loader, test_loader, y_train_np, *,
    alpha_supcon=0.5, beta_kpi=0.3,
    lr_encoder=2e-5, lr_head=1e-3, weight_decay=1e-2,
    epochs=80, patience=15, seed=42,
):
    """Train KAC with uncertainty-weighted KPI contrastive loss."""
    set_seed(seed)
    model = model.to(DEVICE)

    enc_params = [p for p in model.text_encoder.parameters() if p.requires_grad]
    other_params = [p for n, p in model.named_parameters()
                    if p.requires_grad and not n.startswith("text_encoder")]
    optimizer = torch.optim.AdamW([
        {"params": enc_params,   "lr": lr_encoder, "weight_decay": weight_decay},
        {"params": other_params, "lr": lr_head,    "weight_decay": weight_decay},
    ])

    pw = torch.tensor(
        [int((y_train_np == 0).sum()) / max(int((y_train_np == 1).sum()), 1)],
        dtype=torch.float32, device=DEVICE,
    )
    crit_bce = nn.BCEWithLogitsLoss(pos_weight=pw)
    crit_sc  = SupConLoss(temperature=0.07)

    best_f1, best_state, pat_ctr = -1.0, None, 0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        s_bce, s_sc, s_kpi, n = 0., 0., 0., 0

        for batch in train_loader:
            ids   = batch["input_ids"].to(DEVICE)
            mask  = batch["attention_mask"].to(DEVICE)
            resid = batch["resid"].to(DEVICE)
            y     = batch["label"].to(DEVICE)

            logits, z_sc, z_t_kpi, z_r_kpi, _ = model(ids, mask, resid)

            l_bce = crit_bce(logits, y)
            l_sc  = crit_sc(z_sc, y.long())
            l_kpi = kpi_uncertainty_weighted_contrastive_loss(
                z_t_kpi, z_r_kpi, batch["width_raw"].to(DEVICE),
            )

            loss = l_bce + alpha_supcon * l_sc + beta_kpi * l_kpi

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            bs = y.size(0)
            s_bce += l_bce.item()*bs
            s_sc  += l_sc.item()*bs
            s_kpi += l_kpi.item()*bs
            n += bs

        val_m = evaluate(model, val_loader, crit_bce)
        history.append({
            "epoch": epoch,
            "train_bce": s_bce/n, "train_supcon": s_sc/n, "train_kpi": s_kpi/n,
            "val_f1": val_m["f1"], "val_auroc": val_m["auroc"],
        })

        if val_m["f1"] > best_f1:
            best_f1 = val_m["f1"]
            best_state = copy.deepcopy(model.state_dict())
            pat_ctr = 0
        else:
            pat_ctr += 1

        if epoch % 5 == 0 or epoch == 1:
            tag = " *" if pat_ctr == 0 else ""
            print(f"  Ep {epoch:03d} | bce={s_bce/n:.4f} sc={s_sc/n:.4f} kpi={s_kpi/n:.4f} | "
                  f"val_f1={val_m['f1']:.4f} auroc={val_m['auroc']:.4f}{tag}")

        if pat_ctr >= patience:
            print(f"  Early stop at epoch {epoch}")
            break

    model.load_state_dict(best_state)
    test_m = evaluate(model, test_loader, crit_bce)
    return test_m, model, pd.DataFrame(history)


print("Training and evaluation functions defined.")


## 10. Train: KAC with Uncertainty-Weighted KPI Contrastive

In [ ]:
print("=" * 60)
print("Training: KAC + Uncertainty-Weighted KPI Contrastive")
print("=" * 60)

train_loader, val_loader, test_loader = make_loaders()

model = KPIAwareContrastiveModel(
    text_encoder=load_text_encoder(),
    resid_dim=R_train.shape[-1], text_dim=768,
    d_model=64, proj_dim=128,
    n_kpis=N_KPIS, feats_per_kpi=FEATS_PER_KPI,
    num_heads=4, dropout=0.1,
)
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

metrics, trained_model, history = train_kac_uncertainty(
    model, train_loader, val_loader, test_loader, y_train,
    alpha_supcon=0.5, beta_kpi=0.3,
    epochs=80, patience=15, seed=SEED,
)

## 11. Test Results

In [ ]:
print("=" * 60)
print("TEST SET RESULTS — KAC + Uncertainty-Weighted KPI Contrastive")
print("=" * 60)
for k in ["precision", "recall", "f1", "acc", "auroc", "ap"]:
    print(f"  {k:10s}: {metrics[k]:.4f}")

print("\n" + "=" * 60)
print("Classification Report")
print("=" * 60)
print(classification_report(
    metrics["labels"], metrics["preds"],
    target_names=["Normal", "Anomaly"],
))

print("Confusion Matrix:")
cm = confusion_matrix(metrics["labels"], metrics["preds"])
print(cm)

## 12. Multi-Seed Robustness Evaluation

Trains the same method under **5 random seeds** to report **mean ± std** test metrics. This is essential for paper-grade results.

In [ ]:
EVAL_SEEDS = [42, 123, 456, 789, 2024]
_metric_keys = ["precision", "recall", "f1", "acc", "auroc", "ap"]
_rows = []

for run_i, s in enumerate(EVAL_SEEDS):
    print(f"\n{'#' * 60}")
    print(f"# Seed {s} ({run_i + 1}/{len(EVAL_SEEDS)})")
    print(f"{'#' * 60}")
    set_seed(s)
    _tr, _va, _te = make_loaders(seed=s)

    _m = KPIAwareContrastiveModel(
        text_encoder=load_text_encoder(),
        resid_dim=R_train.shape[-1], text_dim=768,
        d_model=64, proj_dim=128,
        n_kpis=N_KPIS, feats_per_kpi=FEATS_PER_KPI,
        num_heads=4, dropout=0.1,
    )
    _mk, _, _ = train_kac_uncertainty(
        _m, _tr, _va, _te, y_train,
        alpha_supcon=0.5, beta_kpi=0.3,
        epochs=80, patience=15, seed=s,
    )
    _rows.append({"seed": s, **{k: _mk[k] for k in _metric_keys}})
    del _m
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

multiseed_df = pd.DataFrame(_rows)

print("\n" + "=" * 60)
print("Per-seed test results")
print("=" * 60)
print(multiseed_df.to_string(index=False))

print("\n" + "=" * 60)
print("Summary: mean ± std (5 seeds)")
print("=" * 60)
for k in _metric_keys:
    mu = multiseed_df[k].mean()
    sd = multiseed_df[k].std(ddof=1)
    print(f"  {k:10s}: {mu:.4f} ± {sd:.4f}")

## 13. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history["epoch"], history["train_bce"], label="BCE")
axes[0].plot(history["epoch"], history["train_supcon"], label="SupCon")
axes[0].plot(history["epoch"], history["train_kpi"], label="KPI (unc-wt)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training Losses")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history["epoch"], history["val_f1"], color="tab:green")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("F1")
axes[1].set_title("Validation F1")
axes[1].grid(True, alpha=0.3)

axes[2].plot(history["epoch"], history["val_auroc"], color="tab:purple")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("AUROC")
axes[2].set_title("Validation AUROC")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: training_curves.png")

## 14. Confusion Matrix Visualization

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    metrics["labels"], metrics["preds"],
    display_labels=["Normal", "Anomaly"],
    cmap="Blues", ax=ax,
)
ax.set_title("Test Confusion Matrix — KAC (Uncertainty-Weighted)")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: confusion_matrix.png")

## 15. ROC and Precision-Recall Curves

In [ ]:
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

RocCurveDisplay.from_predictions(
    metrics["labels"], metrics["probs"],
    name="KAC (unc-wt)", ax=axes[0],
)
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[0].set_title(f"ROC Curve (AUROC = {metrics['auroc']:.4f})")
axes[0].grid(True, alpha=0.3)

PrecisionRecallDisplay.from_predictions(
    metrics["labels"], metrics["probs"],
    name="KAC (unc-wt)", ax=axes[1],
)
axes[1].set_title(f"Precision-Recall Curve (AP = {metrics['ap']:.4f})")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("roc_pr_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: roc_pr_curves.png")

## 16. Save Results

In [ ]:
os.makedirs("results", exist_ok=True)

results_df = pd.DataFrame([{
    "Method": "KAC + Uncertainty-Weighted KPI Contrastive",
    "Precision": metrics["precision"],
    "Recall": metrics["recall"],
    "F1": metrics["f1"],
    "Accuracy": metrics["acc"],
    "AUROC": metrics["auroc"],
    "AP": metrics["ap"],
}])
results_df.to_csv("results/test_results.csv", index=False)
history.to_csv("results/training_history.csv", index=False)

if "multiseed_df" in dir():
    multiseed_df.to_csv("results/multiseed_test_results.csv", index=False)

    summary_rows = []
    for k in _metric_keys:
        summary_rows.append({
            "metric": k,
            "mean": multiseed_df[k].mean(),
            "std": multiseed_df[k].std(ddof=1),
        })
    pd.DataFrame(summary_rows).to_csv("results/multiseed_summary.csv", index=False)

torch.save(trained_model.state_dict(), "results/kac_uncertainty_best.pt")

print("Saved:")
for f in sorted(os.listdir("results")):
    print(f"  results/{f}")

print("\n" + "=" * 60)
print("FINAL RESULTS")
print("=" * 60)
print(f"  F1-Score  : {metrics['f1']:.4f}")
print(f"  AUROC     : {metrics['auroc']:.4f}")
print(f"  Precision : {metrics['precision']:.4f}")
print(f"  Recall    : {metrics['recall']:.4f}")
print(f"  AP        : {metrics['ap']:.4f}")
print(f"  Accuracy  : {metrics['acc']:.4f}")
print("=" * 60)

## 17. Baseline Methods for Comparison

We evaluate several baseline methods on the **same test set** for fair comparison with our KAC + Uncertainty-Weighted method.

| Category | Method | Input |
|----------|--------|-------|
| Unsupervised | Isolation Forest | Raw KPIs (summary stats, 112-D) |
| Unsupervised | LOF | Raw KPIs (summary stats, 112-D) |
| Classical | ARIMA(2,1,0) + LR | ARIMA residual stats (48-D) |
| Tree-based | TimeSeriesForest | Raw KPIs (flattened, 2048-D) |
| Chronos + ML | Chronos-2 + LR | Pre-computed Chronos residuals (aggregated, 240-D) |
| Chronos + ML | Chronos-2 + RF | Pre-computed Chronos residuals (aggregated, 240-D) |
| Rule-based | pct_over_30 | Chronos RAE feature (threshold) |
| Neural | QR-TAN (ctx=20) | Pre-computed Chronos residuals (temporal, 108×80) |

Multi-scale methods (MS-QRF, MS-TAN) are excluded as we have only computed residuals for a single context length (ctx=20).

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler
from scipy import stats as sp_stats

benchmark_results = []

benchmark_results.append({
    "Method": "KAC + Unc-Weighted (Ours)",
    "Precision": metrics["precision"],
    "Recall": metrics["recall"],
    "F1": metrics["f1"],
    "Accuracy": metrics["acc"],
    "AUROC": metrics["auroc"],
    "AP": metrics["ap"],
})
print(f"Stored: KAC + Unc-Weighted (Ours) — F1={metrics['f1']:.4f}, AUROC={metrics['auroc']:.4f}")

### 17.1 Isolation Forest

Unsupervised anomaly detector. Extracts 7 summary statistics per KPI (mean, std, min, max, median, skew, kurtosis) → 112-D feature vector per window.

In [ ]:
from sklearn.ensemble import IsolationForest

def window_summary_features(X_window):
    """7 summary statistics per KPI: mean, std, min, max, median, skew, kurtosis."""
    T, F = X_window.shape
    feats = []
    for j in range(F):
        col = X_window[:, j]
        feats.extend([
            np.mean(col), np.std(col), np.min(col), np.max(col),
            np.median(col), sp_stats.skew(col), sp_stats.kurtosis(col),
        ])
    return np.array(feats)

print("Building Isolation Forest features (7 stats × 16 KPIs = 112-D)...")
X_train_if = np.array([window_summary_features(X_train_raw[i]) for i in range(len(X_train_raw))])
X_test_if  = np.array([window_summary_features(X_test_raw[i])  for i in range(len(X_test_raw))])

iso = IsolationForest(n_estimators=300, contamination=0.5, random_state=42, n_jobs=-1)
iso.fit(X_train_if)

y_score_if = -iso.decision_function(X_test_if)
y_pred_if  = (iso.predict(X_test_if) == -1).astype(int)

p_if, r_if, f1_if, _ = precision_recall_fscore_support(y_test, y_pred_if, average="binary", zero_division=0)
acc_if = accuracy_score(y_test, y_pred_if)
auroc_if = roc_auc_score(y_test, y_score_if)
ap_if = average_precision_score(y_test, y_score_if)

benchmark_results.append({
    "Method": "Isolation Forest",
    "Precision": p_if, "Recall": r_if, "F1": f1_if,
    "Accuracy": acc_if, "AUROC": auroc_if, "AP": ap_if,
})
print(f"Isolation Forest — F1={f1_if:.4f}, AUROC={auroc_if:.4f}")
print(classification_report(y_test, y_pred_if, target_names=["Normal", "Anomaly"]))

### 17.2 Local Outlier Factor (LOF)

Density-based novelty detector. Same 112-D summary features as Isolation Forest, with StandardScaler normalization.

In [ ]:
from sklearn.neighbors import LocalOutlierFactor

X_train_lof = np.array([window_summary_features(X_train_raw[i]) for i in range(len(X_train_raw))])
X_test_lof  = np.array([window_summary_features(X_test_raw[i])  for i in range(len(X_test_raw))])

sc_lof = StandardScaler()
X_train_lof_s = sc_lof.fit_transform(np.nan_to_num(X_train_lof, nan=0.0))
X_test_lof_s  = sc_lof.transform(np.nan_to_num(X_test_lof, nan=0.0))

lof = LocalOutlierFactor(n_neighbors=20, contamination=0.5, novelty=True, n_jobs=-1)
lof.fit(X_train_lof_s)

y_score_lof = -lof.decision_function(X_test_lof_s)
y_pred_lof  = (lof.predict(X_test_lof_s) == -1).astype(int)

p_lof, r_lof, f1_lof, _ = precision_recall_fscore_support(y_test, y_pred_lof, average="binary", zero_division=0)
acc_lof = accuracy_score(y_test, y_pred_lof)
auroc_lof = roc_auc_score(y_test, y_score_lof)
ap_lof = average_precision_score(y_test, y_score_lof)

benchmark_results.append({
    "Method": "LOF",
    "Precision": p_lof, "Recall": r_lof, "F1": f1_lof,
    "Accuracy": acc_lof, "AUROC": auroc_lof, "AP": ap_lof,
})
print(f"LOF — F1={f1_lof:.4f}, AUROC={auroc_lof:.4f}")
print(classification_report(y_test, y_pred_lof, target_names=["Normal", "Anomaly"]))

### 17.3 ARIMA(2,1,0) + Logistic Regression

Classical statistical baseline. Fits ARIMA(2,1,0) per KPI via OLS on first-differenced series, extracts 3 residual statistics per KPI (MAE, std, max_abs) → 48-D feature vector, classifies with balanced Logistic Regression.

In [ ]:
from sklearn.linear_model import LogisticRegression

def arima_residual_features(X_window, p=2):
    """ARIMA(p,1,0) per feature: difference, fit AR(p) by OLS, return residual stats."""
    T, F = X_window.shape
    feats = []
    for j in range(F):
        series = X_window[:, j]
        diff = np.diff(series)
        if len(diff) <= p:
            feats.extend([0.0, 0.0, 0.0])
            continue
        X_ar = np.column_stack([diff[p - i - 1:len(diff) - i - 1] for i in range(p)])
        y_ar = diff[p:]
        try:
            beta, _, _, _ = np.linalg.lstsq(X_ar, y_ar, rcond=None)
            residuals = y_ar - X_ar @ beta
        except Exception:
            residuals = y_ar
        abs_res = np.abs(residuals)
        feats.extend([np.mean(abs_res), np.std(residuals), np.max(abs_res)])
    return np.array(feats)

print("Building ARIMA(2,1,0) residual features (3 stats × 16 KPIs = 48-D)...")
X_train_arima = np.array([arima_residual_features(X_train_raw[i]) for i in range(len(X_train_raw))])
X_test_arima  = np.array([arima_residual_features(X_test_raw[i])  for i in range(len(X_test_raw))])

sc_arima = StandardScaler()
X_train_arima_s = np.nan_to_num(sc_arima.fit_transform(X_train_arima), nan=0.0, posinf=0.0, neginf=0.0)
X_test_arima_s  = np.nan_to_num(sc_arima.transform(X_test_arima), nan=0.0, posinf=0.0, neginf=0.0)

clf_arima = LogisticRegression(class_weight="balanced", max_iter=2000, random_state=42)
clf_arima.fit(X_train_arima_s, y_train)
y_pred_arima = clf_arima.predict(X_test_arima_s)
y_score_arima = clf_arima.predict_proba(X_test_arima_s)[:, 1]

p_ar, r_ar, f1_ar, _ = precision_recall_fscore_support(y_test, y_pred_arima, average="binary", zero_division=0)
acc_ar = accuracy_score(y_test, y_pred_arima)
auroc_ar = roc_auc_score(y_test, y_score_arima)
ap_ar = average_precision_score(y_test, y_score_arima)

benchmark_results.append({
    "Method": "ARIMA(2,1,0) + LR",
    "Precision": p_ar, "Recall": r_ar, "F1": f1_ar,
    "Accuracy": acc_ar, "AUROC": auroc_ar, "AP": ap_ar,
})
print(f"ARIMA(2,1,0) + LR — F1={f1_ar:.4f}, AUROC={auroc_ar:.4f}")
print(classification_report(y_test, y_pred_arima, target_names=["Normal", "Anomaly"]))

### 17.4 TimeSeriesForest

Direct time-series classification using interval-based random forest from the `aeon` library. Raw KPIs are flattened to a single channel of length 2048 (128 × 16).

In [ ]:
try:
    from aeon.classification.interval_based import TimeSeriesForestClassifier
    HAS_AEON = True
except ImportError:
    HAS_AEON = False
    print("WARNING: 'aeon' not installed. Install with: pip install aeon")
    print("Skipping TimeSeriesForest baseline.")

if HAS_AEON:
    n_tr, T_raw, F_raw = X_train_raw.shape
    n_te = X_test_raw.shape[0]
    X_train_flat = X_train_raw.reshape(n_tr, 1, T_raw * F_raw)
    X_test_flat  = X_test_raw.reshape(n_te, 1, T_raw * F_raw)

    tsf = TimeSeriesForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
    print("Fitting TimeSeriesForest (50 estimators)...")
    tsf.fit(X_train_flat, y_train)

    y_pred_tsf = tsf.predict(X_test_flat)
    y_score_tsf = tsf.predict_proba(X_test_flat)[:, 1]

    p_tsf, r_tsf, f1_tsf, _ = precision_recall_fscore_support(y_test, y_pred_tsf, average="binary", zero_division=0)
    acc_tsf = accuracy_score(y_test, y_pred_tsf)
    auroc_tsf = roc_auc_score(y_test, y_score_tsf)
    ap_tsf = average_precision_score(y_test, y_score_tsf)

    benchmark_results.append({
        "Method": "TimeSeriesForest",
        "Precision": p_tsf, "Recall": r_tsf, "F1": f1_tsf,
        "Accuracy": acc_tsf, "AUROC": auroc_tsf, "AP": ap_tsf,
    })
    print(f"TimeSeriesForest — F1={f1_tsf:.4f}, AUROC={auroc_tsf:.4f}")
    print(classification_report(y_test, y_pred_tsf, target_names=["Normal", "Anomaly"]))

### 17.5 Chronos-2 + LR / RF (pre-computed residuals)

Two-stage pipeline: pre-computed Chronos-2 residuals (108 × 80) are aggregated per window using mean, std, and max_abs along the time axis → 240-D feature vector, then classified by Logistic Regression and Random Forest.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

def aggregate_residuals(R):
    """Collapse (N, T, 80) residuals to (N, 240) per-window features: mean, std, max_abs."""
    feat_mean    = R.mean(axis=1)
    feat_std     = R.std(axis=1)
    feat_max_abs = np.abs(R).max(axis=1)
    return np.concatenate([feat_mean, feat_std, feat_max_abs], axis=1)

print("Aggregating Chronos-2 residuals (mean, std, max_abs) → 240-D per window...")
X_train_chr = aggregate_residuals(R_train)
X_test_chr  = aggregate_residuals(R_test)

sc_chr = StandardScaler()
X_train_chr_s = sc_chr.fit_transform(X_train_chr)
X_test_chr_s  = sc_chr.transform(X_test_chr)
print(f"Feature shapes: train {X_train_chr_s.shape}, test {X_test_chr_s.shape}")

for clf_name, clf in [
    ("Chronos-2 + LR", LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=42, n_jobs=-1)),
    ("Chronos-2 + RF", RandomForestClassifier(
        n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1)),
]:
    clf.fit(X_train_chr_s, y_train)
    y_pred_c = clf.predict(X_test_chr_s)
    y_score_c = clf.predict_proba(X_test_chr_s)[:, 1]

    p_c, r_c, f1_c, _ = precision_recall_fscore_support(y_test, y_pred_c, average="binary", zero_division=0)
    acc_c = accuracy_score(y_test, y_pred_c)
    auroc_c = roc_auc_score(y_test, y_score_c)
    ap_c = average_precision_score(y_test, y_score_c)

    benchmark_results.append({
        "Method": clf_name,
        "Precision": p_c, "Recall": r_c, "F1": f1_c,
        "Accuracy": acc_c, "AUROC": auroc_c, "AP": ap_c,
    })
    print(f"\n{clf_name} — F1={f1_c:.4f}, AUROC={auroc_c:.4f}")
    print(classification_report(y_test, y_pred_c, target_names=["Normal", "Anomaly"]))

### 17.6 Rule-Based (pct_over_30)

Chronos-2 rolling forecast rule: for each (timestep, KPI) pair, flag if the Relative Absolute Error (RAE) exceeds 30%. Compute the percentage of flagged points per window, then threshold-optimize on the training set to classify. Reconstructed from the RAE feature (index 4 per KPI block) in pre-computed residuals.

In [ ]:
def compute_pct_over_30(R, X_raw, n_kpis):
    """Reconstruct pct_over_30 from pre-computed Chronos residuals.

    RAE is at feature index 4 within each KPI's 5-feature block.
    Skip near-zero actuals (|actual| < 1e-9) as in the original rule.
    """
    rae_indices = [k * 5 + 4 for k in range(n_kpis)]
    rae = R[:, :, rae_indices]
    T_r = rae.shape[1]
    actuals = X_raw[:, 20:20 + T_r, :]

    valid_mask = np.abs(actuals) >= 1e-9
    over = (rae > 0.30) & valid_mask
    n_over = np.sum(over, axis=(1, 2))
    n_total = np.maximum(np.sum(valid_mask, axis=(1, 2)), 1)
    return n_over / n_total * 100

pct_train = compute_pct_over_30(R_train, X_train_raw, N_KPIS)
pct_test  = compute_pct_over_30(R_test, X_test_raw, N_KPIS)

vals = np.sort(np.unique(pct_train))
candidates = np.concatenate([vals, vals + 1e-9])
best_thr, best_f1_rule = None, -1.0
for thr in candidates:
    y_p = (pct_train >= thr).astype(int)
    tp = np.sum((y_train == 1) & (y_p == 1))
    fp = np.sum((y_train == 0) & (y_p == 1))
    fn = np.sum((y_train == 1) & (y_p == 0))
    pr = tp / (tp + fp + 1e-12)
    rc = tp / (tp + fn + 1e-12)
    f1_v = 2 * pr * rc / (pr + rc + 1e-12)
    if f1_v > best_f1_rule:
        best_f1_rule, best_thr = float(f1_v), float(thr)

print(f"Best threshold (from train): {best_thr:.4f}  (train F1={best_f1_rule:.4f})")

y_pred_rule = (pct_test >= best_thr).astype(int)
y_score_rule = pct_test

p_rule, r_rule, f1_rule, _ = precision_recall_fscore_support(y_test, y_pred_rule, average="binary", zero_division=0)
acc_rule = accuracy_score(y_test, y_pred_rule)
auroc_rule = roc_auc_score(y_test, y_score_rule)
ap_rule = average_precision_score(y_test, y_score_rule)

benchmark_results.append({
    "Method": "Rule-based (pct_over_30)",
    "Precision": p_rule, "Recall": r_rule, "F1": f1_rule,
    "Accuracy": acc_rule, "AUROC": auroc_rule, "AP": ap_rule,
})
print(f"Rule-based — F1={f1_rule:.4f}, AUROC={auroc_rule:.4f}")
print(classification_report(y_test, y_pred_rule, target_names=["Normal", "Anomaly"]))

### 17.7 QR-TAN (ctx=20)

Lightweight temporal attention network operating directly on the Chronos-2 residual time series (108 × 80). Uses a learnable CLS token with multi-head attention to learn which timesteps are most anomalous. Trained with BCE loss + early stopping on validation F1.

In [ ]:
from torch.utils.data import TensorDataset


class QuantileResidualTAN(nn.Module):
    """Lightweight temporal attention over Chronos-2 residual features."""

    def __init__(self, n_features=80, d_model=16, n_heads=2, dropout=0.1):
        super().__init__()
        self.proj = nn.Linear(n_features, d_model)
        self.conv = nn.Conv1d(d_model, d_model, kernel_size=3, padding=1)
        self.norm1 = nn.LayerNorm(d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, 1)

    def forward(self, x):
        B = x.shape[0]
        x = self.proj(x)
        x = self.norm1(x + F.gelu(self.conv(x.transpose(1, 2)).transpose(1, 2)))
        cls = self.cls_token.expand(B, -1, -1)
        attn_out, _ = self.attn(cls, x, x)
        attn_out = self.norm2(attn_out).squeeze(1)
        return self.head(attn_out).squeeze(-1)


sc_qr = StandardScaler()
N_tr_q, T_r_q, F_r_q = R_train.shape
sc_qr.fit(R_train.reshape(-1, F_r_q))
R_train_n = np.nan_to_num(sc_qr.transform(R_train.reshape(-1, F_r_q)).reshape(N_tr_q, T_r_q, F_r_q))
R_val_n   = np.nan_to_num(sc_qr.transform(R_val.reshape(-1, F_r_q)).reshape(len(R_val), T_r_q, F_r_q))
R_test_n  = np.nan_to_num(sc_qr.transform(R_test.reshape(-1, F_r_q)).reshape(len(R_test), T_r_q, F_r_q))

X_tr_qr = torch.tensor(R_train_n, dtype=torch.float32)
y_tr_qr = torch.tensor(y_train, dtype=torch.float32)
X_va_qr = torch.tensor(R_val_n, dtype=torch.float32)
y_va_qr = torch.tensor(y_val, dtype=torch.float32)
X_te_qr = torch.tensor(R_test_n, dtype=torch.float32)

print(f"QR-TAN input: train {X_tr_qr.shape}, val {X_va_qr.shape}, test {X_te_qr.shape}")

set_seed(42)
qr_model = QuantileResidualTAN(n_features=80, d_model=16, n_heads=2, dropout=0.1).to(DEVICE)
qr_opt = torch.optim.AdamW(qr_model.parameters(), lr=1e-3, weight_decay=0.01)
qr_sched = torch.optim.lr_scheduler.CosineAnnealingLR(qr_opt, T_max=150)
pw_qr = torch.tensor([(y_tr_qr == 0).sum() / max((y_tr_qr == 1).sum(), 1)]).to(DEVICE)
qr_crit = nn.BCEWithLogitsLoss(pos_weight=pw_qr)
qr_dl = DataLoader(TensorDataset(X_tr_qr, y_tr_qr), batch_size=32, shuffle=True)

best_qr_f1, best_qr_state, qr_no_improve = 0.0, None, 0

print("Training QR-TAN...")
for epoch in range(200):
    qr_model.train()
    for xb, yb in qr_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        loss = qr_crit(qr_model(xb), yb)
        qr_opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(qr_model.parameters(), 1.0)
        qr_opt.step()
    qr_sched.step()

    qr_model.eval()
    with torch.no_grad():
        vl = qr_model(X_va_qr.to(DEVICE))
        vl_pred = (torch.sigmoid(vl) >= 0.5).cpu().numpy()
        vl_f1 = f1_score(y_val, vl_pred, zero_division=0)

    if (epoch + 1) % 20 == 0:
        print(f"  Ep {epoch+1:3d} | val_f1={vl_f1:.4f}")

    if vl_f1 > best_qr_f1:
        best_qr_f1 = vl_f1
        best_qr_state = copy.deepcopy(qr_model.state_dict())
        qr_no_improve = 0
    else:
        qr_no_improve += 1
        if qr_no_improve >= 20:
            print(f"  Early stop at epoch {epoch+1} (best val F1={best_qr_f1:.4f})")
            break

qr_model.load_state_dict(best_qr_state)
qr_model.eval()
with torch.no_grad():
    te_logits_qr = qr_model(X_te_qr.to(DEVICE))
    te_probs_qr = torch.sigmoid(te_logits_qr).cpu().numpy()
    te_preds_qr = (te_probs_qr >= 0.5).astype(int)

p_qr, r_qr, f1_qr, _ = precision_recall_fscore_support(y_test, te_preds_qr, average="binary", zero_division=0)
acc_qr = accuracy_score(y_test, te_preds_qr)
auroc_qr = roc_auc_score(y_test, te_probs_qr)
ap_qr = average_precision_score(y_test, te_probs_qr)

benchmark_results.append({
    "Method": "QR-TAN (ctx=20)",
    "Precision": p_qr, "Recall": r_qr, "F1": f1_qr,
    "Accuracy": acc_qr, "AUROC": auroc_qr, "AP": ap_qr,
})
print(f"\nQR-TAN — F1={f1_qr:.4f}, AUROC={auroc_qr:.4f}")
print(classification_report(y_test, te_preds_qr, target_names=["Normal", "Anomaly"]))

## 18. Final Comparison Table

Complete benchmark comparing all methods on the balanced TelecomTS test set (494 samples: 247 anomaly, 247 normal). All methods are trained on the same training set and evaluated on the same test set. Sorted by F1-score (descending).

In [ ]:
comparison_df = pd.DataFrame(benchmark_results)
comparison_df = comparison_df.sort_values("F1", ascending=False).reset_index(drop=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.4f}".format)

print("=" * 110)
print("BENCHMARK COMPARISON — Balanced TelecomTS Test Set (494 samples)")
print("=" * 110)
print(comparison_df.to_string(index=False))
print()

for col in ["Precision", "Recall", "F1", "Accuracy", "AUROC", "AP"]:
    best_idx = comparison_df[col].idxmax()
    best_method = comparison_df.loc[best_idx, "Method"]
    best_val = comparison_df.loc[best_idx, col]
    print(f"  Best {col:10s}: {best_val:.4f}  ← {best_method}")

print("\n" + "=" * 110)

comparison_df.to_csv("results/benchmark_comparison.csv", index=False)
print("Saved: results/benchmark_comparison.csv")

### Comparison Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

plot_metrics = ["F1", "AUROC", "AP"]
colors = ["#2196F3" if "(Ours)" not in m else "#FF5722" for m in comparison_df["Method"]]

for ax, metric in zip(axes, plot_metrics):
    bars = ax.barh(
        comparison_df["Method"], comparison_df[metric],
        color=["#FF5722" if "(Ours)" in m else "#90CAF9" for m in comparison_df["Method"]],
        edgecolor="white", linewidth=0.5,
    )
    ax.set_xlabel(metric)
    ax.set_title(f"Test {metric}")
    ax.set_xlim(0, 1.05)
    ax.grid(axis="x", alpha=0.3)
    ax.invert_yaxis()
    for bar, val in zip(bars, comparison_df[metric]):
        ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
                f"{val:.3f}", va="center", fontsize=8)

plt.suptitle("Benchmark Comparison — Balanced TelecomTS", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("results/benchmark_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/benchmark_comparison.png")

## 19. Hyperparameter Tuning (Fast, Frozen Encoder)

**Speed optimization:** Pre-compute DistilBERT hidden states once (frozen encoder), then search over the lightweight fusion head only (~10× faster per epoch).

**Grid:** `alpha_supcon` × `beta_kpi` × `lr_head` = 4 × 4 × 2 = **32 configs**.
Each config: `epochs=30`, `patience=8`, `seed=42`. Best selected by **val AUROC**.
Full training with encoder fine-tuning uses the tuned hyperparameters.

In [ ]:
import itertools
import time as _time

# ── Step A: Pre-compute DistilBERT hidden states (one-time cost) ──────
print("Pre-computing DistilBERT hidden states (frozen encoder) ...")
t_pre = _time.perf_counter()

_encoder = load_text_encoder().to(DEVICE).eval()
for p in _encoder.parameters():
    p.requires_grad = False

@torch.no_grad()
def _precompute_hidden(loader):
    hs, masks, resids, widths, labels = [], [], [], [], []
    for batch in loader:
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        out  = _encoder(input_ids=ids, attention_mask=mask)
        hs.append(out.last_hidden_state.cpu())
        masks.append(mask.cpu())
        resids.append(batch["resid"])
        widths.append(batch["width_raw"])
        labels.append(batch["label"])
    return (torch.cat(hs), torch.cat(masks), torch.cat(resids), torch.cat(widths), torch.cat(labels))

H_train, M_train, R_tr_t, W_tr_t, Y_tr_t = _precompute_hidden(train_loader)
H_val,   M_val,   R_va_t, W_va_t, Y_va_t = _precompute_hidden(val_loader)
del _encoder
if torch.cuda.is_available():
    torch.cuda.empty_cache()
elif hasattr(torch, "mps") and torch.backends.mps.is_available():
    torch.mps.empty_cache()
print(f"  Done in {_time.perf_counter() - t_pre:.1f}s — H_train: {H_train.shape}, H_val: {H_val.shape}")


# ── Step B: Fast dataset + model (no DistilBERT forward pass) ─────────
class _PrecompDataset(Dataset):
    def __init__(self, hidden, mask, resid, width_raw, label):
        self.hidden = hidden
        self.mask = mask
        self.resid = resid
        self.width_raw = width_raw
        self.label = label

    def __len__(self):
        return len(self.label)

    def __getitem__(self, i):
        return {
            "hidden": self.hidden[i],
            "mask": self.mask[i],
            "resid": self.resid[i],
            "width_raw": self.width_raw[i],
            "label": self.label[i],
        }

_ds_tr = _PrecompDataset(H_train, M_train, R_tr_t, W_tr_t, Y_tr_t)
_ds_va = _PrecompDataset(H_val,   M_val,   R_va_t, W_va_t, Y_va_t)
_dl_tr = DataLoader(_ds_tr, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
_dl_va = DataLoader(_ds_va, batch_size=BATCH_SIZE, shuffle=False)

class _KACNoEncoder(nn.Module):
    """KAC fusion head only — takes pre-computed hidden states."""
    def __init__(self, text_dim, resid_dim, d_model, proj_dim,
                 n_kpis, feats_per_kpi, num_heads=4, dropout=0.1):
        super().__init__()
        self.n_kpis, self.feats_per_kpi = n_kpis, feats_per_kpi
        self.text_proj  = nn.Linear(text_dim, d_model)
        self.resid_proj = nn.Linear(resid_dim, d_model)
        self.kpi_queries = nn.Parameter(torch.randn(n_kpis, d_model) * 0.02)
        self.kpi_text_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.resid_kpi_proj = nn.Linear(feats_per_kpi, d_model)
        self.kpi_text_cp  = nn.Linear(d_model, proj_dim)
        self.kpi_resid_cp = nn.Linear(d_model, proj_dim)
        self.supcon_proj = nn.Sequential(nn.Linear(d_model, d_model), nn.ReLU(), nn.Linear(d_model, proj_dim))
        self.gate = nn.Sequential(nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, d_model))
        self.conv = nn.Conv1d(d_model, d_model, 3, padding=1)
        self.drop = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pool_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, d_model), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(d_model, 1))

    def forward(self, hidden_states, attention_mask, x_resid):
        t = self.text_proj(hidden_states); r = self.resid_proj(x_resid); B = t.shape[0]
        kpi_q = self.kpi_queries.unsqueeze(0).expand(B, -1, -1)
        kpi_text, _ = self.kpi_text_attn(query=kpi_q, key=t, value=t, key_padding_mask=(attention_mask == 0))
        kpi_resid_list = []
        for k in range(self.n_kpis):
            s, e = k * self.feats_per_kpi, (k + 1) * self.feats_per_kpi
            kpi_resid_list.append(self.resid_kpi_proj(x_resid[:, :, s:e]).mean(dim=1))
        kpi_resid = torch.stack(kpi_resid_list, dim=1)
        z_t_kpi = F.normalize(self.kpi_text_cp(kpi_text), dim=-1)
        z_r_kpi = F.normalize(self.kpi_resid_cp(kpi_resid), dim=-1)
        mask_f = attention_mask.unsqueeze(-1).float()
        t_pool = (t * mask_f).sum(1) / mask_f.sum(1).clamp(min=1)
        gate = torch.sigmoid(self.gate(t_pool)).unsqueeze(1)
        x = torch.cat([t, r * gate], dim=1)
        x = self.norm(x + self.drop(self.conv(x.transpose(1, 2)).transpose(1, 2)))
        cls = self.cls_token.expand(B, -1, -1)
        pooled, _ = self.pool_attn(query=cls, key=x, value=x)
        pooled = pooled.squeeze(1)
        z_sc = F.normalize(self.supcon_proj(pooled), dim=-1)
        logits = self.head(pooled).squeeze(-1)
        return logits, z_sc, z_t_kpi, z_r_kpi


# ── Step C: Fast train (val-only, frozen encoder, no test eval) ───────
def _train_kac_fast(train_dl, val_dl, y_train_np, *,
                    alpha_supcon, beta_kpi, lr_head, kpi_mode="uncertainty",
                    weight_decay=1e-2, epochs=30, patience=8, seed=42):
    set_seed(seed)
    model = _KACNoEncoder(
        text_dim=768, resid_dim=R_train.shape[-1], d_model=64,
        proj_dim=128, n_kpis=N_KPIS, feats_per_kpi=FEATS_PER_KPI,
        num_heads=4, dropout=0.1,
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr_head, weight_decay=weight_decay)
    pw = torch.tensor([(y_train_np == 0).sum() / max((y_train_np == 1).sum(), 1)],
                       dtype=torch.float32, device=DEVICE)
    crit_bce = nn.BCEWithLogitsLoss(pos_weight=pw)
    crit_sc  = SupConLoss(temperature=0.07)
    best_f1, best_auroc, pat_ctr = -1.0, -1.0, 0

    for epoch in range(1, epochs + 1):
        model.train()
        for batch in train_dl:
            h     = batch["hidden"].to(DEVICE)
            m     = batch["mask"].to(DEVICE)
            resid = batch["resid"].to(DEVICE)
            y     = batch["label"].to(DEVICE)
            logits, z_sc, z_t_kpi, z_r_kpi = model(h, m, resid)
            l_bce = crit_bce(logits, y)
            l_sc  = crit_sc(z_sc, y.long()) if alpha_supcon > 0 else torch.tensor(0., device=DEVICE)
            if beta_kpi > 0:
                if kpi_mode == "uncertainty":
                    l_kpi = kpi_uncertainty_weighted_contrastive_loss(z_t_kpi, z_r_kpi, batch["width_raw"].to(DEVICE))
                else:
                    l_kpi = kpi_aware_contrastive_loss(z_t_kpi, z_r_kpi)
            else:
                l_kpi = torch.tensor(0., device=DEVICE)
            loss = l_bce + alpha_supcon * l_sc + beta_kpi * l_kpi
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()

        model.eval()
        with torch.no_grad():
            all_lo, all_la = [], []
            for batch in val_dl:
                lo = model(batch["hidden"].to(DEVICE), batch["mask"].to(DEVICE), batch["resid"].to(DEVICE))[0]
                all_lo.append(lo.cpu()); all_la.append(batch["label"])
            al = torch.cat(all_lo).numpy(); yl = torch.cat(all_la).numpy()
            probs = 1.0 / (1.0 + np.exp(-al))
            preds = (probs >= 0.5).astype(int)
            _, _, f1_v, _ = precision_recall_fscore_support(yl, preds, average="binary", zero_division=0)
            try:    auroc_v = roc_auc_score(yl, probs)
            except: auroc_v = float("nan")

        if f1_v > best_f1: best_f1 = f1_v; pat_ctr = 0
        else: pat_ctr += 1
        if auroc_v > best_auroc: best_auroc = auroc_v
        if pat_ctr >= patience: break

    return {"val_f1": best_f1, "val_auroc": best_auroc, "stopped_epoch": epoch}


# ── Step D: Grid search ──────────────────────────────────────────────
HP_GRID = list(itertools.product(
    [0.1, 0.3, 0.5, 1.0],     # alpha_supcon
    [0.1, 0.3, 0.5, 1.0],     # beta_kpi
    [5e-4, 1e-3],              # lr_head
))
TUNE_SEED = 42

print(f"\nHP tuning (frozen encoder): {len(HP_GRID)} configs × seed={TUNE_SEED}, "
      f"epochs=30, patience=8\n")

t0 = _time.perf_counter()
tune_results = []

for i, (a_sc, b_kpi, lr_h) in enumerate(HP_GRID, 1):
    m = _train_kac_fast(
        _dl_tr, _dl_va, y_train,
        alpha_supcon=a_sc, beta_kpi=b_kpi, lr_head=lr_h,
        kpi_mode="uncertainty", epochs=30, patience=8, seed=TUNE_SEED,
    )
    tune_results.append({
        "alpha_supcon": a_sc, "beta_kpi": b_kpi, "lr_head": lr_h,
        "val_auroc": m["val_auroc"], "val_f1": m["val_f1"],
        "stopped_ep": m["stopped_epoch"],
    })
    if i % 8 == 0 or i == len(HP_GRID):
        elapsed = _time.perf_counter() - t0
        eta = elapsed / i * (len(HP_GRID) - i)
        print(f"  {i:3d}/{len(HP_GRID)} done  ({elapsed:.0f}s elapsed, ~{eta:.0f}s remaining)")

df_tune = pd.DataFrame(tune_results).sort_values("val_auroc", ascending=False)
print(f"\nCompleted in {_time.perf_counter() - t0:.0f}s\n")

print("Top 10 configurations (ranked by val AUROC):")
print(df_tune[["alpha_supcon", "beta_kpi", "lr_head", "val_auroc", "val_f1", "stopped_ep"]].head(10).to_string(index=False))

best = df_tune.iloc[0]
BEST_ALPHA = best["alpha_supcon"]
BEST_BETA  = best["beta_kpi"]
BEST_LR_H  = best["lr_head"]

print(f"\n{'='*60}")
print(f"BEST CONFIG (by val AUROC):")
print(f"  alpha_supcon = {BEST_ALPHA}")
print(f"  beta_kpi     = {BEST_BETA}")
print(f"  lr_head      = {BEST_LR_H}")
print(f"  val_auroc    = {best['val_auroc']:.4f}")
print(f"  val_f1       = {best['val_f1']:.4f}")
print(f"{'='*60}")

df_tune.to_csv("results/hp_tuning_grid.csv", index=False)
print("Saved: results/hp_tuning_grid.csv")


## 20. Ablation Study — Three KAC Variants × 5 Seeds

Isolate the contribution of each loss component by comparing three variants:

| Variant | BCE | SupCon | KPI Contrastive |
|---------|-----|--------|-----------------|
| **KAC — BCE only** | ✓ | ✗ | ✗ |
| **KAC — Standard KPI** | ✓ | ✓ | Standard (uniform weight) |
| **KAC — Uncertainty-Weighted (Ours)** | ✓ | ✓ | Uncertainty-weighted |

All variants use the **same architecture** (DistilBERT + Chronos residuals + gated fusion).
Tuned hyperparameters (`alpha_supcon`, `beta_kpi`, `lr_head`) from Section 19.
Test probabilities are collected for the threshold sweep in Section 22.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

SEEDS = [42, 123, 456, 789, 1337]
BATCH_SIZE = 16  # ensure consistent

# ── General-purpose KAC training (handles all 3 loss variants) ────────
def train_kac_general(
    train_loader, val_loader, test_loader, y_train_np, *,
    kpi_mode="uncertainty",  # "none" | "standard" | "uncertainty"
    alpha_supcon=0.5, beta_kpi=0.3,
    lr_encoder=2e-5, lr_head=1e-3, weight_decay=1e-2,
    epochs=80, patience=15, seed=42, verbose=False,
):
    """Train KAC and return test metrics + test probabilities."""
    set_seed(seed)
    model = KPIAwareContrastiveModel(
        text_encoder=load_text_encoder(),
        resid_dim=R_train.shape[-1], text_dim=768,
        d_model=64, proj_dim=128, n_kpis=N_KPIS,
        feats_per_kpi=FEATS_PER_KPI, num_heads=4, dropout=0.1,
    ).to(DEVICE)

    enc_params = [p for p in model.text_encoder.parameters() if p.requires_grad]
    other_params = [p for n, p in model.named_parameters()
                    if p.requires_grad and not n.startswith("text_encoder")]
    optimizer = torch.optim.AdamW([
        {"params": enc_params,   "lr": lr_encoder, "weight_decay": weight_decay},
        {"params": other_params, "lr": lr_head,    "weight_decay": weight_decay},
    ])

    pw = torch.tensor([(y_train_np == 0).sum() / max((y_train_np == 1).sum(), 1)],
                       dtype=torch.float32, device=DEVICE)
    crit_bce = nn.BCEWithLogitsLoss(pos_weight=pw)
    crit_sc  = SupConLoss(temperature=0.07)

    best_f1, best_state, pat_ctr = -1.0, None, 0

    for epoch in range(1, epochs + 1):
        model.train()
        for batch in train_loader:
            ids   = batch["input_ids"].to(DEVICE)
            mask  = batch["attention_mask"].to(DEVICE)
            resid = batch["resid"].to(DEVICE)
            y     = batch["label"].to(DEVICE)
            logits, z_sc, z_t_kpi, z_r_kpi, _ = model(ids, mask, resid)
            l_bce = crit_bce(logits, y)

            if alpha_supcon > 0 and kpi_mode != "none":
                l_sc = crit_sc(z_sc, y.long())
            else:
                l_sc = torch.tensor(0., device=DEVICE)

            if beta_kpi > 0 and kpi_mode == "uncertainty":
                l_kpi = kpi_uncertainty_weighted_contrastive_loss(z_t_kpi, z_r_kpi, batch["width_raw"].to(DEVICE))
            elif beta_kpi > 0 and kpi_mode == "standard":
                l_kpi = kpi_aware_contrastive_loss(z_t_kpi, z_r_kpi)
            else:
                l_kpi = torch.tensor(0., device=DEVICE)

            loss = l_bce + alpha_supcon * l_sc + beta_kpi * l_kpi
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()

        val_m = evaluate(model, val_loader, crit_bce)
        if val_m["f1"] > best_f1:
            best_f1 = val_m["f1"]; best_state = copy.deepcopy(model.state_dict()); pat_ctr = 0
        else:
            pat_ctr += 1
        if pat_ctr >= patience:
            if verbose: print(f"    Early stop at epoch {epoch}")
            break

    model.load_state_dict(best_state)
    test_m = evaluate(model, test_loader, crit_bce)
    test_m["stopped_epoch"] = epoch

    model.eval()
    with torch.no_grad():
        lo, la = [], []
        for batch in test_loader:
            lo.append(model(batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE),
                            batch["resid"].to(DEVICE))[0].cpu())
            la.append(batch["label"])
        logits_all = torch.cat(lo).numpy()
        labels_all = torch.cat(la).numpy()
    probs = 1.0 / (1.0 + np.exp(-logits_all))

    return test_m, probs, labels_all


# ── Define ablation variants ──────────────────────────────────────────
ABLATION_METHODS = {
    "KAC — BCE only": {
        "kpi_mode": "none", "alpha_supcon": 0.0, "beta_kpi": 0.0, "lr_head": BEST_LR_H,
        "color": "#ff7f0e", "marker": "s",
    },
    "KAC — Standard KPI": {
        "kpi_mode": "standard", "alpha_supcon": BEST_ALPHA, "beta_kpi": BEST_BETA, "lr_head": BEST_LR_H,
        "color": "#1f77b4", "marker": "o",
    },
    "KAC — Uncertainty-Weighted (Ours)": {
        "kpi_mode": "uncertainty", "alpha_supcon": BEST_ALPHA, "beta_kpi": BEST_BETA, "lr_head": BEST_LR_H,
        "color": "#2ca02c", "marker": "^",
    },
}

# ── Run ablation: 3 variants × 5 seeds ───────────────────────────────
ablation_results = {}  # method → list of per-seed dicts
ablation_probs   = {}  # method → list of (probs, labels)

t0_ablation = _time.perf_counter()

for method_name, cfg in ABLATION_METHODS.items():
    print(f"\n{'='*60}")
    print(f"{method_name} × {len(SEEDS)} seeds")
    print(f"  kpi_mode={cfg['kpi_mode']}, alpha={cfg['alpha_supcon']}, beta={cfg['beta_kpi']}, lr={cfg['lr_head']}")
    print(f"{'='*60}")

    ablation_results[method_name] = []
    ablation_probs[method_name] = []

    for seed in SEEDS:
        metrics, probs, labels = train_kac_general(
            train_loader, val_loader, test_loader, y_train,
            kpi_mode=cfg["kpi_mode"],
            alpha_supcon=cfg["alpha_supcon"], beta_kpi=cfg["beta_kpi"],
            lr_encoder=2e-5, lr_head=cfg["lr_head"], weight_decay=1e-2,
            epochs=80, patience=15, seed=seed, verbose=False,
        )
        ablation_results[method_name].append({"seed": seed, **metrics})
        ablation_probs[method_name].append((probs, labels))
        print(f"  seed {seed}: F1={metrics['f1']:.4f}  AUROC={metrics['auroc']:.4f}  "
              f"Prec={metrics['precision']:.4f}  Rec={metrics['recall']:.4f}  AP={metrics['ap']:.4f}")

t_ablation = _time.perf_counter() - t0_ablation
print(f"\n>>> Ablation completed in {t_ablation:.1f}s ({len(ABLATION_METHODS)*len(SEEDS)} runs)")

# ── Per-method summaries ──────────────────────────────────────────────
for method_name in ABLATION_METHODS:
    df_m = pd.DataFrame(ablation_results[method_name])
    print(f"\n{method_name} — 5-Seed Summary:")
    for col in ["f1", "auroc", "precision", "recall", "ap"]:
        print(f"  {col:12s}: {df_m[col].mean():.4f} ± {df_m[col].std():.4f}")


## 21. Three-Way Ablation Comparison Table

Paper-ready table showing the contribution of each loss component (5 seeds, mean ± std).

In [ ]:
metrics_cols = ["f1", "auroc", "ap", "precision", "recall"]

rows_abl = []
for method_name in ABLATION_METHODS:
    df_m = pd.DataFrame(ablation_results[method_name])
    row = {"Method": method_name}
    for m in metrics_cols:
        row[m] = f"{df_m[m].mean():.4f} ± {df_m[m].std():.4f}"
    rows_abl.append(row)

df_ablation_table = pd.DataFrame(rows_abl)

print("\n" + "=" * 110)
print("THREE-WAY ABLATION — TelecomTS Balanced (5 seeds)")
print(f"Tuned HPs: alpha_supcon={BEST_ALPHA}, beta_kpi={BEST_BETA}, lr_head={BEST_LR_H}")
print("=" * 110)
print(df_ablation_table.to_string(index=False))
print("=" * 110)

df_bce  = pd.DataFrame(ablation_results["KAC — BCE only"])
df_std  = pd.DataFrame(ablation_results["KAC — Standard KPI"])
df_uw   = pd.DataFrame(ablation_results["KAC — Uncertainty-Weighted (Ours)"])

print(f"\n★ Uncertainty-Weighted vs BCE-only:    "
      f"F1 {df_uw['f1'].mean() - df_bce['f1'].mean():+.4f}  |  "
      f"AUROC {df_uw['auroc'].mean() - df_bce['auroc'].mean():+.4f}")
print(f"★ Uncertainty-Weighted vs Standard KPI: "
      f"F1 {df_uw['f1'].mean() - df_std['f1'].mean():+.4f}  |  "
      f"AUROC {df_uw['auroc'].mean() - df_std['auroc'].mean():+.4f}")
print(f"★ Standard KPI vs BCE-only:             "
      f"F1 {df_std['f1'].mean() - df_bce['f1'].mean():+.4f}  |  "
      f"AUROC {df_std['auroc'].mean() - df_bce['auroc'].mean():+.4f}")

df_ablation_table.to_csv("results/ablation_3way_comparison.csv", index=False)
for name, df_s in [("bce_only", df_bce), ("standard_kpi", df_std), ("uncertainty_weighted", df_uw)]:
    df_s.to_csv(f"results/ablation_{name}_per_seed.csv", index=False)
print("\nSaved: results/ablation_*.csv")

## 22. Threshold Sensitivity Analysis (Parallelized)

Sweep decision thresholds from 0.0 to 1.0 for all three ablation variants.
Uses **vectorized numpy** per seed + **ThreadPoolExecutor** across seed-method pairs.

Note: AUROC and AP are threshold-independent (horizontal lines in the plot).

In [ ]:
THRESHOLDS = np.linspace(0, 1, 21)  # 0.00, 0.05, ..., 1.00
METRIC_NAMES = ["f1", "auroc", "precision", "recall", "ap"]

def _sweep_one_seed(probs, labels, thresholds):
    """Vectorized threshold sweep for a single seed."""
    T = len(thresholds)
    preds = (probs[None, :] >= thresholds[:, None]).astype(np.int32)
    y = labels[None, :]
    tp = (preds * y).sum(axis=1).astype(float)
    fp = (preds * (1 - y)).sum(axis=1).astype(float)
    fn = ((1 - preds) * y).sum(axis=1).astype(float)
    prec = np.where(tp + fp > 0, tp / (tp + fp), 0.0)
    rec  = np.where(tp + fn > 0, tp / (tp + fn), 0.0)
    f1   = np.where(prec + rec > 0, 2 * prec * rec / (prec + rec), 0.0)
    try:    auroc = roc_auc_score(labels, probs)
    except: auroc = np.nan
    try:    ap = average_precision_score(labels, probs)
    except: ap = np.nan
    return {"precision": prec, "recall": rec, "f1": f1,
            "auroc": np.full(T, auroc), "ap": np.full(T, ap)}

# ── Parallel sweep using pre-collected test probabilities ─────────────
print("Computing threshold metrics in parallel …")
t1 = _time.perf_counter()

tasks = []; task_keys = []
for method_name in ABLATION_METHODS:
    for seed_idx, (probs, labels) in enumerate(ablation_probs[method_name]):
        tasks.append((probs, labels, THRESHOLDS))
        task_keys.append((method_name, seed_idx))

n_workers = min(len(tasks), os.cpu_count() or 4)
with ThreadPoolExecutor(max_workers=n_workers) as executor:
    futures = {executor.submit(_sweep_one_seed, *t): k for t, k in zip(tasks, task_keys)}
    sweep_raw = {m: [] for m in ABLATION_METHODS}
    for fut in as_completed(futures):
        method, _ = futures[fut]
        sweep_raw[method].append(fut.result())

sweep = {}
for method in ABLATION_METHODS:
    sweep[method] = {}
    for m in METRIC_NAMES:
        arr = np.stack([s[m] for s in sweep_raw[method]])
        sweep[method][m] = (np.nanmean(arr, axis=0), np.nanstd(arr, axis=0))

t_sweep = _time.perf_counter() - t1
print(f">>> Threshold sweep: {t_sweep*1000:.1f}ms  "
      f"({len(tasks)} seed-method pairs × {len(THRESHOLDS)} thresholds, {n_workers} workers)")

# ── Plot ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(28, 5.5))

for ax, metric in zip(axes, METRIC_NAMES):
    for method_name, cfg in ABLATION_METHODS.items():
        mean, std = sweep[method_name][metric]
        ax.plot(THRESHOLDS, mean, color=cfg["color"], marker=cfg["marker"],
                linewidth=2, markersize=4, label=method_name)
        ax.fill_between(THRESHOLDS, np.maximum(mean - std, 0.0),
                        np.minimum(mean + std, 1.0), color=cfg["color"], alpha=0.15)
    ax.set_xlabel("Decision Threshold", fontsize=12)
    ax.set_ylabel(metric.upper(), fontsize=12)
    ax.set_title(metric.upper(), fontsize=14, fontweight="bold")
    ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.05)
    ax.set_xticks(np.arange(0, 1.1, 0.1))
    ax.grid(True, alpha=0.3, linestyle="--")
    ax.legend(fontsize=7, loc="best")

fig.suptitle("TelecomTS Balanced — Metrics vs Decision Threshold  (5 seeds, mean ± SD)",
             fontsize=16, fontweight="bold", y=1.03)
plt.tight_layout()
plt.savefig("results/threshold_sweep_ablation.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/threshold_sweep_ablation.png")

# ── Save sweep data as CSV ────────────────────────────────────────────
rows_sw = []
for method in ABLATION_METHODS:
    for i, thr in enumerate(THRESHOLDS):
        row = {"method": method, "threshold": round(float(thr), 3)}
        for m in METRIC_NAMES:
            mean, std = sweep[method][m]
            row[f"{m}_mean"] = round(float(mean[i]), 4)
            row[f"{m}_std"]  = round(float(std[i]), 4)
        rows_sw.append(row)
pd.DataFrame(rows_sw).to_csv("results/threshold_sweep_data.csv", index=False)
print("Saved: results/threshold_sweep_data.csv")

## 23. Paper-Ready Summary — All Methods

Unified table combining:
- **Ablation variants** (3 KAC variants, 5-seed mean ± std)
- **Baseline methods** (from Section 17, single-run)

Sorted by F1-score.

In [ ]:
# ── Combine ablation (mean ± std) + baselines (single run) ────────────
paper_rows = []

# Ablation variants (multi-seed)
for method_name in ABLATION_METHODS:
    df_m = pd.DataFrame(ablation_results[method_name])
    row = {
        "Method": method_name,
        "F1": f"{df_m['f1'].mean():.4f} ± {df_m['f1'].std():.4f}",
        "AUROC": f"{df_m['auroc'].mean():.4f} ± {df_m['auroc'].std():.4f}",
        "AP": f"{df_m['ap'].mean():.4f} ± {df_m['ap'].std():.4f}",
        "Precision": f"{df_m['precision'].mean():.4f} ± {df_m['precision'].std():.4f}",
        "Recall": f"{df_m['recall'].mean():.4f} ± {df_m['recall'].std():.4f}",
        "f1_sort": df_m['f1'].mean(),
        "Seeds": "5",
    }
    paper_rows.append(row)

# Baselines from benchmark_results (Section 17)
if 'benchmark_results' in dir():
    for br in benchmark_results:
        if "KAC" in br.get("Method", ""):
            continue
        row = {
            "Method": br["Method"],
            "F1": f"{br['F1']:.4f}",
            "AUROC": f"{br['AUROC']:.4f}",
            "AP": f"{br['AP']:.4f}",
            "Precision": f"{br['Precision']:.4f}",
            "Recall": f"{br['Recall']:.4f}",
            "f1_sort": br["F1"],
            "Seeds": "1",
        }
        paper_rows.append(row)

df_paper = pd.DataFrame(paper_rows).sort_values("f1_sort", ascending=False).drop(columns=["f1_sort"])

print("\n" + "=" * 130)
print("PAPER-READY SUMMARY — TelecomTS Balanced Dataset")
print("=" * 130)
print(df_paper.to_string(index=False))
print("=" * 130)

df_paper.to_csv("results/paper_summary_all_methods.csv", index=False)
print("\nSaved: results/paper_summary_all_methods.csv")

# ── Bar chart: all methods ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
methods_plot = df_paper["Method"].tolist()
n_methods = len(methods_plot)

for ax, col, title in zip(axes, ["F1", "AUROC", "AP"], ["F1-Score", "AUROC", "Average Precision"]):
    means, errs = [], []
    for _, row in df_paper.iterrows():
        val = row[col]
        if "±" in str(val):
            parts = val.split("±")
            means.append(float(parts[0].strip()))
            errs.append(float(parts[1].strip()))
        else:
            means.append(float(val))
            errs.append(0.0)

    colors = []
    for _, row in df_paper.iterrows():
        name = row["Method"]
        if "Uncertainty" in name:
            colors.append("#2ca02c")
        elif "Standard KPI" in name:
            colors.append("#1f77b4")
        elif "BCE only" in name:
            colors.append("#ff7f0e")
        else:
            colors.append("#9467bd")

    bars = ax.barh(range(n_methods), means, xerr=errs, color=colors, alpha=0.85, edgecolor="white")
    ax.set_yticks(range(n_methods))
    ax.set_yticklabels(methods_plot, fontsize=8)
    ax.set_xlabel(title, fontsize=12)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlim(0, 1.05)
    ax.grid(True, axis="x", alpha=0.3, linestyle="--")
    ax.invert_yaxis()

fig.suptitle("TelecomTS Balanced — All Methods Comparison", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("results/paper_all_methods_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/paper_all_methods_comparison.png")

## 24. Ablation Bar Chart (Paper Figure)

Side-by-side grouped bar chart of the three KAC ablation variants (5 seeds, mean ± std) — suitable for paper figures.

In [ ]:
ablation_names = list(ABLATION_METHODS.keys())
ablation_colors = [ABLATION_METHODS[n]["color"] for n in ablation_names]
metrics_fig = ["f1", "auroc", "ap", "precision", "recall"]
metric_labels = ["F1", "AUROC", "AP", "Precision", "Recall"]

n_metrics = len(metrics_fig)
n_variants = len(ablation_names)
x = np.arange(n_metrics)
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))

for i, (name, color) in enumerate(zip(ablation_names, ablation_colors)):
    df_m = pd.DataFrame(ablation_results[name])
    means = [df_m[m].mean() for m in metrics_fig]
    stds  = [df_m[m].std()  for m in metrics_fig]
    ax.bar(x + i * width, means, width, yerr=stds, label=name,
           color=color, alpha=0.85, edgecolor="white", capsize=3)

ax.set_xticks(x + width)
ax.set_xticklabels(metric_labels, fontsize=12)
ax.set_ylabel("Score", fontsize=12)
ax.set_ylim(0, 1.08)
ax.set_title("TelecomTS — Ablation: KAC Loss Component Contribution  (5 seeds, mean ± SD)",
             fontsize=14, fontweight="bold")
ax.legend(fontsize=9, loc="lower right")
ax.grid(True, axis="y", alpha=0.3, linestyle="--")

plt.tight_layout()
plt.savefig("results/ablation_bar_chart.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/ablation_bar_chart.png")

## 25. Save All Artifacts

Export all results, CSVs, and model metadata to `results/` for paper reproducibility.

In [ ]:
import json

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

# ── Hyperparameter config ─────────────────────────────────────────────
hp_config = {
    "best_alpha_supcon": float(BEST_ALPHA),
    "best_beta_kpi": float(BEST_BETA),
    "best_lr_head": float(BEST_LR_H),
    "seeds": SEEDS,
    "hp_grid_size": len(HP_GRID),
    "dataset": "TelecomTS Balanced",
    "train_samples": len(y_train),
    "val_samples": len(y_val),
    "test_samples": len(y_test),
    "n_kpis": N_KPIS,
    "feats_per_kpi": FEATS_PER_KPI,
    "resid_shape": list(R_train.shape),
}
with open(results_dir / "experiment_config.json", "w") as f:
    json.dump(hp_config, f, indent=2)

# ── LaTeX-ready ablation table ────────────────────────────────────────
latex_rows = []
for method_name in ABLATION_METHODS:
    df_m = pd.DataFrame(ablation_results[method_name])
    short = method_name.replace("KAC — ", "").replace(" (Ours)", "")
    vals = " & ".join(f"${df_m[m].mean():.3f} \\pm {df_m[m].std():.3f}$" for m in ["f1", "auroc", "ap", "precision", "recall"])
    latex_rows.append(f"  {short} & {vals} \\\\")

latex_table = "\\begin{table}[t]\n\\centering\n\\caption{Ablation study on TelecomTS (5 seeds, mean $\\pm$ std).}\n"
latex_table += "\\label{tab:ablation_telecomts}\n"
latex_table += "\\begin{tabular}{lccccc}\n\\toprule\n"
latex_table += "Variant & F1 & AUROC & AP & Precision & Recall \\\\\n\\midrule\n"
latex_table += "\n".join(latex_rows) + "\n\\bottomrule\n\\end{tabular}\n\\end{table}"

with open(results_dir / "ablation_table.tex", "w") as f:
    f.write(latex_table)

print("All artifacts saved to results/:")
for p in sorted(results_dir.glob("*")):
    print(f"  {p.name}")
print(f"\nLaTeX table preview:\n{latex_table}")

## 26. ROC & Precision-Recall Curves — KAC vs Baselines

Side-by-side ROC and PR curves comparing:
- **KAC + Uncertainty-Weighted (Ours)** — mean ± std over 5 seeds (shaded band)
- **Chronos-2 + LR** — single deterministic run
- **Rule-based (pct_over_30)** — single deterministic run

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, auc

# ── 1. KAC Uncertainty-Weighted: 5 seeds ─────────────────────────────
kac_seed_data = ablation_probs["KAC — Uncertainty-Weighted (Ours)"]

# ── 2. Chronos-2 + LR: re-fit (y_score_c was overwritten by RF loop) ─
lr_clf = LogisticRegression(
    class_weight="balanced", max_iter=1000, random_state=42, n_jobs=-1)
lr_clf.fit(X_train_chr_s, y_train)
lr_scores = lr_clf.predict_proba(X_test_chr_s)[:, 1]

# ── 3. Rule-based scores (pct_test still in scope) ───────────────────
rule_scores = pct_test

# ── Style ─────────────────────────────────────────────────────────────
COLORS = {"KAC (Ours)": "#2ca02c", "Chronos-2 + LR": "#ff7f0e",
          "Rule-based": "#7f4fbf"}
N_PTS = 200

# ────────────────────────────────────────────────────────────────────
#  ROC curves
# ────────────────────────────────────────────────────────────────────
base_fpr = np.linspace(0, 1, N_PTS)
kac_tprs, kac_aurocs = [], []
for probs, labels in kac_seed_data:
    fpr_i, tpr_i, _ = roc_curve(labels, probs)
    kac_tprs.append(np.interp(base_fpr, fpr_i, tpr_i))
    kac_tprs[-1][0] = 0.0
    kac_aurocs.append(auc(fpr_i, tpr_i))

kac_tpr_mean = np.mean(kac_tprs, axis=0)
kac_tpr_std  = np.std(kac_tprs, axis=0)
kac_auroc_mean, kac_auroc_std = np.mean(kac_aurocs), np.std(kac_aurocs)

fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_scores)
auroc_lr = auc(fpr_lr, tpr_lr)

fpr_rb, tpr_rb, _ = roc_curve(y_test, rule_scores)
auroc_rb = auc(fpr_rb, tpr_rb)

# ────────────────────────────────────────────────────────────────────
#  PR curves
# ────────────────────────────────────────────────────────────────────
base_recall = np.linspace(0, 1, N_PTS)
kac_precs, kac_aps = [], []
for probs, labels in kac_seed_data:
    prec_i, rec_i, _ = precision_recall_curve(labels, probs)
    kac_precs.append(np.interp(base_recall, rec_i[::-1], prec_i[::-1]))
    kac_aps.append(average_precision_score(labels, probs))

kac_prec_mean = np.mean(kac_precs, axis=0)
kac_prec_std  = np.std(kac_precs, axis=0)
kac_ap_mean, kac_ap_std = np.mean(kac_aps), np.std(kac_aps)

prec_lr, rec_lr, _ = precision_recall_curve(y_test, lr_scores)
ap_lr = average_precision_score(y_test, lr_scores)

prec_rb, rec_rb, _ = precision_recall_curve(y_test, rule_scores)
ap_rb = average_precision_score(y_test, rule_scores)

# ────────────────────────────────────────────────────────────────────
#  Plot
# ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- ROC ---
ax = axes[0]
ax.plot(base_fpr, kac_tpr_mean, color=COLORS["KAC (Ours)"], lw=2.5,
        label=f"KAC (Ours) (AUC = {kac_auroc_mean:.3f} $\\pm$ {kac_auroc_std:.3f})")
ax.fill_between(base_fpr,
                np.clip(kac_tpr_mean - kac_tpr_std, 0, 1),
                np.clip(kac_tpr_mean + kac_tpr_std, 0, 1),
                color=COLORS["KAC (Ours)"], alpha=0.15)
ax.plot(fpr_lr, tpr_lr, color=COLORS["Chronos-2 + LR"], lw=2,
        label=f"Chronos-2 + LR (AUC = {auroc_lr:.3f})")
ax.plot(fpr_rb, tpr_rb, color=COLORS["Rule-based"], lw=2,
        label=f"Rule-based (AUC = {auroc_rb:.3f})")
ax.plot([0, 1], [0, 1], "k--", alpha=0.3, lw=1)
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curve", fontsize=14)
ax.legend(loc="lower right", fontsize=10, framealpha=0.9)
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.01, 1.01)
ax.set_ylim(-0.01, 1.05)

# --- PR ---
ax = axes[1]
ax.plot(base_recall, kac_prec_mean, color=COLORS["KAC (Ours)"], lw=2.5,
        label=f"KAC (Ours) (AP = {kac_ap_mean:.3f} $\\pm$ {kac_ap_std:.3f})")
ax.fill_between(base_recall,
                np.clip(kac_prec_mean - kac_prec_std, 0, 1),
                np.clip(kac_prec_mean + kac_prec_std, 0, 1),
                color=COLORS["KAC (Ours)"], alpha=0.15)
ax.plot(rec_lr, prec_lr, color=COLORS["Chronos-2 + LR"], lw=2,
        label=f"Chronos-2 + LR (AP = {ap_lr:.3f})")
ax.plot(rec_rb, prec_rb, color=COLORS["Rule-based"], lw=2,
        label=f"Rule-based (AP = {ap_rb:.3f})")
ax.set_xlabel("Recall", fontsize=12)
ax.set_ylabel("Precision", fontsize=12)
ax.set_title("Precision-Recall Curve", fontsize=14)
ax.legend(loc="lower left", fontsize=10, framealpha=0.9)
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.01, 1.01)
ax.set_ylim(-0.01, 1.05)

plt.tight_layout()
plt.savefig("results/roc_pr_3methods.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved: results/roc_pr_3methods.png")

## 27. Grouped Bar Chart — All 8 Methods × F1 / AUROC / AP

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

methods_order = [
    "KAC + Unc-Weighted (Ours)",
    "QR-TAN (ctx=20)",
    "Chronos-2 + RF",
    "Chronos-2 + LR",
    "ARIMA(2,1,0) + LR",
    "Rule-based (pct_over_30)",
    "LOF",
    "Isolation Forest",
]

df_bar = comparison_df.set_index("Method").loc[methods_order]

short_names = [
    "KAC (Ours)", "QR-TAN", "Chronos+RF", "Chronos+LR",
    "ARIMA+LR", "Rule-based", "LOF", "Iso. Forest",
]

metric_cols = ["F1", "AUROC", "AP"]
metric_colors = ["#2ca02c", "#1f77b4", "#ff7f0e"]

x = np.arange(len(short_names))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 5.5))

for i, (col, color) in enumerate(zip(metric_cols, metric_colors)):
    vals = df_bar[col].values
    bars = ax.bar(x + (i - 1) * width, vals, width, label=col, color=color,
                  edgecolor="white", linewidth=0.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.008,
                f"{v:.3f}", ha="center", va="bottom", fontsize=7, rotation=90)

ax.set_xticks(x)
ax.set_xticklabels(short_names, fontsize=10, rotation=25, ha="right")
ax.set_ylabel("Score", fontsize=12)
ax.set_title("TelecomTS Benchmark — F1 / AUROC / AP (8 methods)", fontsize=14)
ax.set_ylim(0, 1.12)
ax.legend(fontsize=11, loc="upper right")
ax.grid(axis="y", alpha=0.3)
ax.axhline(y=1.0, color="grey", linestyle=":", linewidth=0.7, alpha=0.5)

plt.tight_layout()
plt.savefig("results/bar_chart_8methods.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved: results/bar_chart_8methods.png")

## 28. Zoomed-In ROC — Top-Left Region (FPR < 0.2)

Focus on the operationally relevant region where FPR is low. Shows the top 4 methods (KAC, QR-TAN, Chronos-2+RF, Chronos-2+LR) where the differences are visually meaningful.

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib as _mpl
_mpl.rcParams["axes.facecolor"] = "white"
_mpl.rcParams["figure.facecolor"] = "white"
_mpl.rcParams["savefig.facecolor"] = "white"

# ── Re-fit Chronos-2 + LR (y_score_c was overwritten by RF in the loop) ──
lr_clf_ = LogisticRegression(
    class_weight="balanced", max_iter=1000, random_state=42, n_jobs=-1)
lr_clf_.fit(X_train_chr_s, y_train)
lr_scores_ = lr_clf_.predict_proba(X_test_chr_s)[:, 1]

# ── Re-fit Chronos-2 + RF to get scores ──
from sklearn.ensemble import RandomForestClassifier as _RFC
rf_clf_ = _RFC(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1)
rf_clf_.fit(X_train_chr_s, y_train)
rf_scores_ = rf_clf_.predict_proba(X_test_chr_s)[:, 1]

# ── Curves for single-run methods ──────────────────────────────────
methods_zoom = {
    "KAC":     {"color": "#2ca02c", "lw": 1.8},
    "QR-TAN":         {"color": "#d62728", "lw": 1.4},
    "Chronos-2 + RF": {"color": "#1f77b4", "lw": 1.4},
    "Chronos-2 + LR": {"color": "#ff7f0e", "lw": 1.4},
}

single_scores = {
    "QR-TAN":         te_probs_qr,
    "Chronos-2 + RF": rf_scores_,
    "Chronos-2 + LR": lr_scores_,
}

N_PTS = 300
base_fpr = np.linspace(0, 1, N_PTS)

fig, axes = plt.subplots(1, 2, figsize=(7, 3.2), facecolor="white")

# ── Left: full ROC with zoom box ──────────────────────────────────
ax_full = axes[0]
ax_full.set_facecolor("white")

# KAC (5-seed mean ± std)
kac_tprs = []
kac_aurocs = []
for probs, labels in ablation_probs["KAC — Uncertainty-Weighted (Ours)"]:
    fpr_i, tpr_i, _ = roc_curve(labels, probs)
    kac_tprs.append(np.interp(base_fpr, fpr_i, tpr_i))
    kac_tprs[-1][0] = 0.0
    kac_aurocs.append(auc(fpr_i, tpr_i))

kac_mean = np.mean(kac_tprs, axis=0)
kac_std  = np.std(kac_tprs, axis=0)
kac_auc_m, kac_auc_s = np.mean(kac_aurocs), np.std(kac_aurocs)

ax_full.plot(base_fpr, kac_mean, color="#2ca02c", lw=1.8,
             label=f"KAC (AUC = {kac_auc_m:.3f} ± {kac_auc_s:.3f})")
ax_full.fill_between(base_fpr,
                     np.clip(kac_mean - kac_std, 0, 1),
                     np.clip(kac_mean + kac_std, 0, 1),
                     color="#2ca02c", alpha=0.12)

for name, scores in single_scores.items():
    fpr_s, tpr_s, _ = roc_curve(y_test, scores)
    auc_s = auc(fpr_s, tpr_s)
    ax_full.plot(fpr_s, tpr_s, color=methods_zoom[name]["color"],
                 lw=methods_zoom[name]["lw"],
                 label=f"{name} (AUC = {auc_s:.3f})")

ax_full.plot([0, 1], [0, 1], "k--", alpha=0.3, lw=1)
from matplotlib.patches import Rectangle
ax_full.add_patch(Rectangle((0, 0.8), 0.2, 0.2, fill=False,
                              edgecolor="grey", linestyle="--", lw=1.0))
ax_full.set_xlabel("False Positive Rate", fontsize=9)
ax_full.set_ylabel("True Positive Rate", fontsize=9)
ax_full.set_title("ROC Curve (Full)", fontsize=10)
ax_full.legend(loc="lower right", fontsize=6.5, framealpha=0.9)
ax_full.tick_params(axis="both", labelsize=8)
#ax_full.grid(True, alpha=0.3)

# ── Right: zoomed ROC (FPR < 0.2, TPR > 0.8) ─────────────────────
ax_zoom = axes[1]
ax_zoom.set_facecolor("white")

fpr_mask = base_fpr <= 0.2
ax_zoom.plot(base_fpr[fpr_mask], kac_mean[fpr_mask], color="#2ca02c", lw=1.8,
             label=f"KAC")
ax_zoom.fill_between(base_fpr[fpr_mask],
                     np.clip(kac_mean[fpr_mask] - kac_std[fpr_mask], 0, 1),
                     np.clip(kac_mean[fpr_mask] + kac_std[fpr_mask], 0, 1),
                     color="#2ca02c", alpha=0.15)

for name, scores in single_scores.items():
    fpr_s, tpr_s, _ = roc_curve(y_test, scores)
    mask = fpr_s <= 0.2
    ax_zoom.plot(fpr_s[mask], tpr_s[mask], color=methods_zoom[name]["color"],
                 lw=methods_zoom[name]["lw"], label=name)

ax_zoom.set_xlim(-0.005, 0.205)
ax_zoom.set_ylim(0.79, 1.01)
ax_zoom.set_xlabel("False Positive Rate", fontsize=9)
ax_zoom.set_ylabel("True Positive Rate", fontsize=9)
ax_zoom.set_title("ROC Curve (Zoomed: FPR ≤ 0.2)", fontsize=10)
ax_zoom.legend(loc="lower right", fontsize=6.5, framealpha=0.9)
ax_zoom.tick_params(axis="both", labelsize=8)
#ax_zoom.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("results/roc_zoomed_top4.pdf", bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.savefig("results/roc_zoomed_top4.png", dpi=300, bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.show()
print("Saved: results/roc_zoomed_top4.pdf")
print("Saved: results/roc_zoomed_top4.png")

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib as _mpl
_mpl.rcParams["axes.facecolor"] = "white"
_mpl.rcParams["figure.facecolor"] = "white"
_mpl.rcParams["savefig.facecolor"] = "white"

# ── Re-fit Chronos-2 + LR (y_score_c was overwritten by RF in the loop) ──
lr_clf_ = LogisticRegression(
    class_weight="balanced", max_iter=1000, random_state=42, n_jobs=-1)
lr_clf_.fit(X_train_chr_s, y_train)
lr_scores_ = lr_clf_.predict_proba(X_test_chr_s)[:, 1]

# ── Re-fit Chronos-2 + RF to get scores ──
from sklearn.ensemble import RandomForestClassifier as _RFC
rf_clf_ = _RFC(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1)
rf_clf_.fit(X_train_chr_s, y_train)
rf_scores_ = rf_clf_.predict_proba(X_test_chr_s)[:, 1]

# ── Curves for single-run methods ──────────────────────────────────
methods_zoom = {
    "KAC":     {"color": "#2ca02c", "lw": 1.8},
    "QR-TAN":         {"color": "#d62728", "lw": 1.4},
    "Chronos-2 + RF": {"color": "#1f77b4", "lw": 1.4},
    "Chronos-2 + LR": {"color": "#ff7f0e", "lw": 1.4},
}

single_scores = {
    "QR-TAN":         te_probs_qr,
    "Chronos-2 + RF": rf_scores_,
    "Chronos-2 + LR": lr_scores_,
}

N_PTS = 300
base_fpr = np.linspace(0, 1, N_PTS)

fig, axes = plt.subplots(1, 2, figsize=(7, 3.2), facecolor="white")

# ── Left: full ROC with zoom box ──────────────────────────────────
ax_full = axes[0]
ax_full.set_facecolor("white")

# KAC (5-seed mean ± std)
kac_tprs = []
kac_aurocs = []
for probs, labels in ablation_probs["KAC — Uncertainty-Weighted (Ours)"]:
    fpr_i, tpr_i, _ = roc_curve(labels, probs)
    kac_tprs.append(np.interp(base_fpr, fpr_i, tpr_i))
    kac_tprs[-1][0] = 0.0
    kac_aurocs.append(auc(fpr_i, tpr_i))

kac_mean = np.mean(kac_tprs, axis=0)
kac_std  = np.std(kac_tprs, axis=0)
kac_auc_m, kac_auc_s = np.mean(kac_aurocs), np.std(kac_aurocs)

ax_full.plot(base_fpr, kac_mean, color="#2ca02c", lw=1.8,
             label=f"KAC (AUC = {kac_auc_m:.3f} ± {kac_auc_s:.3f})")
ax_full.fill_between(base_fpr,
                     np.clip(kac_mean - kac_std, 0, 1),
                     np.clip(kac_mean + kac_std, 0, 1),
                     color="#2ca02c", alpha=0.12)

for name, scores in single_scores.items():
    fpr_s, tpr_s, _ = roc_curve(y_test, scores)
    auc_s = auc(fpr_s, tpr_s)
    ax_full.plot(fpr_s, tpr_s, color=methods_zoom[name]["color"],
                 lw=methods_zoom[name]["lw"],
                 label=f"{name} (AUC = {auc_s:.3f})")

ax_full.plot([0, 1], [0, 1], "k--", alpha=0.3, lw=1)
from matplotlib.patches import Rectangle
ax_full.add_patch(Rectangle((0, 0.8), 0.2, 0.2, fill=False,
                              edgecolor="grey", linestyle="--", lw=1.0))
ax_full.set_xlabel("False Positive Rate", fontsize=9)
ax_full.set_ylabel("True Positive Rate", fontsize=9)
ax_full.set_title("ROC Curve (Full)", fontsize=10)
ax_full.legend(loc="lower right", fontsize=6.5, framealpha=0.9)
ax_full.tick_params(axis="both", labelsize=8)
#ax_full.grid(True, alpha=0.3)

# ── Right: zoomed ROC (FPR < 0.2, TPR > 0.8) ─────────────────────
ax_zoom = axes[1]
ax_zoom.set_facecolor("white")

fpr_mask = base_fpr <= 0.2
ax_zoom.plot(base_fpr[fpr_mask], kac_mean[fpr_mask], color="#2ca02c", lw=1.8,
             label=f"KAC")
ax_zoom.fill_between(base_fpr[fpr_mask],
                     np.clip(kac_mean[fpr_mask] - kac_std[fpr_mask], 0, 1),
                     np.clip(kac_mean[fpr_mask] + kac_std[fpr_mask], 0, 1),
                     color="#2ca02c", alpha=0.15)

for name, scores in single_scores.items():
    fpr_s, tpr_s, _ = roc_curve(y_test, scores)
    mask = fpr_s <= 0.2
    ax_zoom.plot(fpr_s[mask], tpr_s[mask], color=methods_zoom[name]["color"],
                 lw=methods_zoom[name]["lw"], label=name)

ax_zoom.set_xlim(-0.005, 0.205)
ax_zoom.set_ylim(0.79, 1.01)
ax_zoom.set_xlabel("False Positive Rate", fontsize=9)
ax_zoom.set_ylabel("True Positive Rate", fontsize=9)
ax_zoom.set_title("ROC Curve (Zoomed: FPR ≤ 0.2)", fontsize=10)
ax_zoom.legend(loc="lower right", fontsize=6.5, framealpha=0.9)
ax_zoom.tick_params(axis="both", labelsize=8)
#ax_zoom.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("results/roc_zoomed_top4.pdf", bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.savefig("results/roc_zoomed_top4.png", dpi=300, bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.show()
print("Saved: results/roc_zoomed_top4.pdf")
print("Saved: results/roc_zoomed_top4.png")

## 29. ROC & PR Curves — KAC vs QR-TAN vs Chronos-2+LR

Three competitive methods with tighter AUC spread (0.965 – 0.989), making visual differences more meaningful than the previous figure with Rule-based.

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, auc

COLORS3 = {"KAC (Ours)": "#2ca02c", "QR-TAN": "#d62728", "Chronos-2 + LR": "#ff7f0e"}
N_PTS = 300

kac_seed_data = ablation_probs["KAC — Uncertainty-Weighted (Ours)"]

# ──────────────────────────────────────────────────────────────────
#  Compute all ROC curves
# ──────────────────────────────────────────────────────────────────
base_fpr = np.linspace(0, 1, N_PTS)

kac_tprs, kac_aurocs = [], []
for probs, labels in kac_seed_data:
    fpr_i, tpr_i, _ = roc_curve(labels, probs)
    kac_tprs.append(np.interp(base_fpr, fpr_i, tpr_i))
    kac_tprs[-1][0] = 0.0
    kac_aurocs.append(auc(fpr_i, tpr_i))
kac_tpr_m = np.mean(kac_tprs, axis=0)
kac_tpr_s = np.std(kac_tprs, axis=0)
kac_auc_m, kac_auc_s = np.mean(kac_aurocs), np.std(kac_aurocs)

fpr_qr, tpr_qr, _ = roc_curve(y_test, te_probs_qr)
auc_qr = auc(fpr_qr, tpr_qr)

fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_scores_)
auc_lr = auc(fpr_lr, tpr_lr)

# ──────────────────────────────────────────────────────────────────
#  Compute all PR curves
# ──────────────────────────────────────────────────────────────────
base_recall = np.linspace(0, 1, N_PTS)

kac_precs, kac_aps = [], []
for probs, labels in kac_seed_data:
    prec_i, rec_i, _ = precision_recall_curve(labels, probs)
    kac_precs.append(np.interp(base_recall, rec_i[::-1], prec_i[::-1]))
    kac_aps.append(average_precision_score(labels, probs))
kac_prec_m = np.mean(kac_precs, axis=0)
kac_prec_s = np.std(kac_precs, axis=0)
kac_ap_m, kac_ap_s = np.mean(kac_aps), np.std(kac_aps)

prec_qr, rec_qr, _ = precision_recall_curve(y_test, te_probs_qr)
ap_qr_val = average_precision_score(y_test, te_probs_qr)

prec_lr, rec_lr, _ = precision_recall_curve(y_test, lr_scores_)
ap_lr_val = average_precision_score(y_test, lr_scores_)

# ──────────────────────────────────────────────────────────────────
#  Plot
# ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- ROC ---
ax = axes[0]
ax.plot(base_fpr, kac_tpr_m, color=COLORS3["KAC (Ours)"], lw=2.5,
        label=f"KAC (Ours) (AUC = {kac_auc_m:.3f} ± {kac_auc_s:.3f})")
ax.fill_between(base_fpr,
                np.clip(kac_tpr_m - kac_tpr_s, 0, 1),
                np.clip(kac_tpr_m + kac_tpr_s, 0, 1),
                color=COLORS3["KAC (Ours)"], alpha=0.15)
ax.plot(fpr_qr, tpr_qr, color=COLORS3["QR-TAN"], lw=2,
        label=f"QR-TAN (AUC = {auc_qr:.3f})")
ax.plot(fpr_lr, tpr_lr, color=COLORS3["Chronos-2 + LR"], lw=2,
        label=f"Chronos-2 + LR (AUC = {auc_lr:.3f})")
ax.plot([0, 1], [0, 1], "k--", alpha=0.3, lw=1)
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curve", fontsize=14)
ax.legend(loc="lower right", fontsize=10, framealpha=0.9)
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.01, 1.01); ax.set_ylim(-0.01, 1.05)

# --- PR ---
ax = axes[1]
ax.plot(base_recall, kac_prec_m, color=COLORS3["KAC (Ours)"], lw=2.5,
        label=f"KAC (Ours) (AP = {kac_ap_m:.3f} ± {kac_ap_s:.3f})")
ax.fill_between(base_recall,
                np.clip(kac_prec_m - kac_prec_s, 0, 1),
                np.clip(kac_prec_m + kac_prec_s, 0, 1),
                color=COLORS3["KAC (Ours)"], alpha=0.15)
ax.plot(rec_qr, prec_qr, color=COLORS3["QR-TAN"], lw=2,
        label=f"QR-TAN (AP = {ap_qr_val:.3f})")
ax.plot(rec_lr, prec_lr, color=COLORS3["Chronos-2 + LR"], lw=2,
        label=f"Chronos-2 + LR (AP = {ap_lr_val:.3f})")
ax.set_xlabel("Recall", fontsize=12)
ax.set_ylabel("Precision", fontsize=12)
ax.set_title("Precision-Recall Curve", fontsize=14)
ax.legend(loc="lower left", fontsize=10, framealpha=0.9)
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.01, 1.01); ax.set_ylim(-0.01, 1.05)

plt.tight_layout()
plt.savefig("results/roc_pr_kac_qrtan_chronos.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved: results/roc_pr_kac_qrtan_chronos.png")

## 30. Modality Ablation — Text-Only vs Residual-Only vs Full KAC

**Goal:** Demonstrate that neither modality alone is sufficient — the multimodal fusion of Chronos residuals and text descriptions is essential for the high performance of KAC.

**Setup (3 variants × 5 seeds):**

| Variant | Text Input | Residual Input | Training Losses |
|---------|-----------|---------------|-----------------|
| **Text-Only** | Real descriptions | Zeroed (all 0) | BCE only |
| **Residual-Only** | Dummy token (".") | Real Chronos residuals | BCE only |
| **KAC + Unc-Weighted (Ours)** | Real descriptions | Real Chronos residuals | BCE + SupCon + KPI Contrastive |

- Text-Only disables KPI contrastive since residual embeddings are meaningless with zero input.
- Residual-Only uses an uninformative single-token text so DistilBERT produces constant embeddings, isolating the residual signal.
- Full method reuses existing results from Section 20.

In [ ]:
import os, time as _time, threading
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from concurrent.futures import ThreadPoolExecutor, as_completed

torch.set_num_threads(4)

# ── (A) Text-Only data: real text, zeroed residuals ──────────────
R_zero_train = np.zeros_like(R_train)
R_zero_val   = np.zeros_like(R_val)
R_zero_test  = np.zeros_like(R_test)

# ── (B) Residual-Only data: dummy text, real residuals ───────────
_dummy_tr = tokenizer(["."] * len(y_train), padding="max_length",
                       truncation=True, max_length=MAX_LEN, return_tensors="pt")
_dummy_va = tokenizer(["."] * len(y_val),   padding="max_length",
                       truncation=True, max_length=MAX_LEN, return_tensors="pt")
_dummy_te = tokenizer(["."] * len(y_test),  padding="max_length",
                       truncation=True, max_length=MAX_LEN, return_tensors="pt")

_MOD_CONFIGS = {
    "Text-Only (No Residuals)": dict(
        tr_ids=train_tokens["input_ids"], tr_mask=train_tokens["attention_mask"], tr_resid=R_zero_train,
        va_ids=val_tokens["input_ids"],   va_mask=val_tokens["attention_mask"],   va_resid=R_zero_val,
        te_ids=test_tokens["input_ids"],  te_mask=test_tokens["attention_mask"],  te_resid=R_zero_test,
    ),
    "Residual-Only (No Text)": dict(
        tr_ids=_dummy_tr["input_ids"], tr_mask=_dummy_tr["attention_mask"], tr_resid=R_train,
        va_ids=_dummy_va["input_ids"], va_mask=_dummy_va["attention_mask"], va_resid=R_val,
        te_ids=_dummy_te["input_ids"], te_mask=_dummy_te["attention_mask"], te_resid=R_test,
    ),
}

SEEDS = [42, 123, 456, 789, 1337]
N_WORKERS = 2
_print_lock = threading.Lock()

def _train_one(vname, seed, dcfg):
    """Train one (variant, seed) in a thread. PyTorch releases the GIL
    during tensor ops, so two threads genuinely run in parallel on CPU."""
    g = torch.Generator(); g.manual_seed(seed)
    tr_ld = DataLoader(ContrastiveDataset(dcfg["tr_ids"], dcfg["tr_mask"], dcfg["tr_resid"], y_train),
                       batch_size=BATCH_SIZE, shuffle=True, generator=g)
    va_ld = DataLoader(ContrastiveDataset(dcfg["va_ids"], dcfg["va_mask"], dcfg["va_resid"], y_val),
                       batch_size=BATCH_SIZE, shuffle=False)
    te_ld = DataLoader(ContrastiveDataset(dcfg["te_ids"], dcfg["te_mask"], dcfg["te_resid"], y_test),
                       batch_size=BATCH_SIZE, shuffle=False)

    metrics, probs, labels = train_kac_general(
        tr_ld, va_ld, te_ld, y_train,
        kpi_mode="none", alpha_supcon=0.0, beta_kpi=0.0,
        lr_head=BEST_LR_H, seed=seed, verbose=False,
    )
    with _print_lock:
        print(f"  {vname} seed {seed}: F1={metrics['f1']:.4f}  AUROC={metrics['auroc']:.4f}  "
              f"Prec={metrics['precision']:.4f}  Rec={metrics['recall']:.4f}  AP={metrics['ap']:.4f}")
    return vname, seed, metrics, probs, labels

# ── Launch parallel training ─────────────────────────────────────
task_args = [(vname, seed, dcfg)
             for vname, dcfg in _MOD_CONFIGS.items()
             for seed in SEEDS]

t0 = _time.perf_counter()
print(f"Launching {len(task_args)} training jobs ({N_WORKERS} threads, "
      f"torch intra-op threads={torch.get_num_threads()})")
print(f"Expected wall time: ~{len(task_args) * 30 // N_WORKERS} min "
      f"(vs ~{len(task_args) * 30} min serial)\n")

raw_results = []
with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
    futures = {executor.submit(_train_one, *args): args for args in task_args}
    for f in as_completed(futures):
        raw_results.append(f.result())

elapsed = _time.perf_counter() - t0
print(f"\n>>> Modality ablation completed in {elapsed:.1f}s "
      f"({len(task_args)} models, {N_WORKERS} threads)")

# ── Collect results ──────────────────────────────────────────────
modality_results = {}
modality_probs   = {}
for vname, seed, metrics, probs, labels in raw_results:
    modality_results.setdefault(vname, []).append(metrics)
    modality_probs.setdefault(vname, []).append((probs, labels))

# ── Reuse existing full-method results ───────────────────────────
_FULL_KEY = "KAC — Uncertainty-Weighted (Ours)"
modality_results["KAC + Unc-Weighted (Ours)"] = ablation_results[_FULL_KEY]
modality_probs["KAC + Unc-Weighted (Ours)"]   = ablation_probs[_FULL_KEY]

# ── Summary table ────────────────────────────────────────────────
_ORDER = ["KAC + Unc-Weighted (Ours)", "Both Modalities (BCE Only)", "Residual-Only (No Text)", "Text-Only (No Residuals)"]
_MCOLS = ["f1", "auroc", "precision", "recall", "ap"]

print("\n" + "="*110)
print("MODALITY ABLATION — TelecomTS (balanced, 494 test, 5-seed mean ± std)")
print("="*110)
rows_mod = []
for name in _ORDER:
    df_m = pd.DataFrame(modality_results[name])
    row = {"Method": name}
    for m in _MCOLS:
        row[f"{m}_mean"] = df_m[m].mean()
        row[f"{m}_std"]  = df_m[m].std()
    rows_mod.append(row)
    print(f"  {name:40s}  F1={row['f1_mean']:.3f}±{row['f1_std']:.3f}  "
          f"AUROC={row['auroc_mean']:.3f}±{row['auroc_std']:.3f}  "
          f"Prec={row['precision_mean']:.3f}±{row['precision_std']:.3f}  "
          f"Rec={row['recall_mean']:.3f}±{row['recall_std']:.3f}  "
          f"AP={row['ap_mean']:.3f}±{row['ap_std']:.3f}")

modality_df = pd.DataFrame(rows_mod)

# ── LaTeX table ──────────────────────────────────────────────────
_TEX_ORDER = ["f1", "auroc", "precision", "recall", "ap"]
_TEX_HEADS = ["F1", "AUROC", "Prec.", "Rec.", "AP"]

print("\n── LaTeX (copy-paste into paper) ──────────────────────────")
print(r"\begin{table}[t]")
print(r"  \caption{Modality ablation on TelecomTS (balanced test set, 494 samples, mean $\pm$ std over 5 seeds). Best in \textbf{bold}.}")
print(r"  \label{tab:telecomts_modality}")
print(r"  \small")
print(r"  \begin{tabular}{lccccc}")
print(r"    \toprule")
hdr = " & ".join([r"\textbf{" + h + "}" for h in ["Method"] + _TEX_HEADS])
print(f"    {hdr} \\\\")
print(r"    \midrule")

for row in rows_mod:
    name_tex = row["Method"]
    if "(Ours)" in name_tex:
        name_tex = name_tex.replace("(Ours)", r"(\textbf{Ours})")
    name_tex = name_tex.replace("_", r"\_")
    vals = []
    for m in _TEX_ORDER:
        mn, sd = row[f"{m}_mean"], row[f"{m}_std"]
        best_mn = max(r[f"{m}_mean"] for r in rows_mod)
        cell = f".{mn:.3f}"[1:] + r"\tiny{$\pm$" + f".{sd:.3f}"[1:] + "}"
        if abs(mn - best_mn) < 1e-6:
            cell = r"\textbf{" + cell + "}"
        vals.append(cell)
    print(f"    {name_tex} & {' & '.join(vals)} \\\\")

print(r"    \bottomrule")
print(r"  \end{tabular}")
print(r"\end{table}")

## 31. Disentangled Ablation — Fusion vs Loss Design

**Goal:** Separate the contributions of (a) multimodal fusion and (b) contrastive loss design by adding a "Both Modalities + BCE Only" baseline.

| Comparison | What it measures |
|---|---|
| Both Modalities (BCE Only) vs single-modality | Pure **fusion** contribution |
| KAC + Unc-Weighted (Ours) vs Both Modalities (BCE Only) | Pure **loss design** contribution (SupCon + KPI) |

In [ ]:
import time as _time, threading
from concurrent.futures import ThreadPoolExecutor, as_completed

torch.set_num_threads(4)

# ── Both Modalities (BCE Only): real text + real residuals, no contrastive losses
_BCE_CFG = dict(
    tr_ids=train_tokens["input_ids"], tr_mask=train_tokens["attention_mask"], tr_resid=R_train,
    va_ids=val_tokens["input_ids"],   va_mask=val_tokens["attention_mask"],   va_resid=R_val,
    te_ids=test_tokens["input_ids"],  te_mask=test_tokens["attention_mask"],  te_resid=R_test,
)

SEEDS = [42, 123, 456, 789, 1337]
N_WORKERS = 2
_print_lock = threading.Lock()

def _train_bce_only(seed):
    g = torch.Generator(); g.manual_seed(seed)
    tr_ld = DataLoader(ContrastiveDataset(_BCE_CFG["tr_ids"], _BCE_CFG["tr_mask"], _BCE_CFG["tr_resid"], y_train),
                       batch_size=BATCH_SIZE, shuffle=True, generator=g)
    va_ld = DataLoader(ContrastiveDataset(_BCE_CFG["va_ids"], _BCE_CFG["va_mask"], _BCE_CFG["va_resid"], y_val),
                       batch_size=BATCH_SIZE, shuffle=False)
    te_ld = DataLoader(ContrastiveDataset(_BCE_CFG["te_ids"], _BCE_CFG["te_mask"], _BCE_CFG["te_resid"], y_test),
                       batch_size=BATCH_SIZE, shuffle=False)
    metrics, probs, labels = train_kac_general(
        tr_ld, va_ld, te_ld, y_train,
        kpi_mode="none", alpha_supcon=0.0, beta_kpi=0.0,
        lr_head=BEST_LR_H, seed=seed, verbose=False,
    )
    with _print_lock:
        print(f"  Both Modalities (BCE Only) seed {seed}: F1={metrics['f1']:.4f}  "
              f"AUROC={metrics['auroc']:.4f}  Prec={metrics['precision']:.4f}  "
              f"Rec={metrics['recall']:.4f}  AP={metrics['ap']:.4f}")
    return seed, metrics, probs, labels

t0 = _time.perf_counter()
print(f"Launching 5 'Both Modalities (BCE Only)' jobs ({N_WORKERS} threads)\n")

bce_results, bce_probs = [], []
with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
    futures = {executor.submit(_train_bce_only, s): s for s in SEEDS}
    for f in as_completed(futures):
        seed, metrics, probs, labels = f.result()
        bce_results.append(metrics)
        bce_probs.append((probs, labels))

elapsed = _time.perf_counter() - t0
print(f"\n>>> Completed in {elapsed:.1f}s")

# ── Build full 4-row comparison table ────────────────────────────
_FULL_KEY = "KAC — Uncertainty-Weighted (Ours)"
all_results = {
    "KAC + Unc-Weighted (Ours)":   ablation_results[_FULL_KEY],
    "Both Modalities (BCE Only)":  bce_results,
    "Residual-Only (No Text)":     modality_results["Residual-Only (No Text)"],
    "Text-Only (No Residuals)":    modality_results["Text-Only (No Residuals)"],
}

_ORDER = list(all_results.keys())
_MCOLS = ["f1", "auroc", "precision", "recall", "ap"]

print("\n" + "="*120)
print("DISENTANGLED ABLATION — TelecomTS (balanced, 494 test, 5-seed mean ± std)")
print("="*120)
rows = []
for name in _ORDER:
    df_m = pd.DataFrame(all_results[name])
    row = {"Method": name}
    for m in _MCOLS:
        row[f"{m}_mean"] = df_m[m].mean()
        row[f"{m}_std"]  = df_m[m].std()
    rows.append(row)
    print(f"  {name:40s}  F1={row['f1_mean']:.3f}±{row['f1_std']:.3f}  "
          f"AUROC={row['auroc_mean']:.3f}±{row['auroc_std']:.3f}  "
          f"Prec={row['precision_mean']:.3f}±{row['precision_std']:.3f}  "
          f"Rec={row['recall_mean']:.3f}±{row['recall_std']:.3f}  "
          f"AP={row['ap_mean']:.3f}±{row['ap_std']:.3f}")

# ── Contribution breakdown ───────────────────────────────────────
ours_f1 = rows[0]["f1_mean"]
bce_f1  = rows[1]["f1_mean"]
resid_f1 = rows[2]["f1_mean"]
text_f1  = rows[3]["f1_mean"]
best_single = max(resid_f1, text_f1)

print(f"\n── Contribution breakdown (F1) ──")
print(f"  Fusion gain (BCE-both vs best single-modality):  +{(bce_f1 - best_single)*100:.1f}pp")
print(f"  Loss design gain (Ours vs BCE-both):             +{(ours_f1 - bce_f1)*100:.1f}pp")
print(f"  Total gain (Ours vs best single-modality):       +{(ours_f1 - best_single)*100:.1f}pp")

# ── LaTeX table ──────────────────────────────────────────────────
_TEX_HEADS = ["F1", "AUROC", "Prec.", "Rec.", "AP"]

print("\n── LaTeX (copy-paste into paper) ──────────────────────────")
print(r"\begin{table}[t]")
print(r"  \caption{Disentangled ablation on TelecomTS (balanced, 494 test samples, mean $\pm$ std over 5 seeds).}")
print(r"  \label{tab:telecomts_disentangled}")
print(r"  \small")
print(r"  \begin{tabular}{lccccc}")
print(r"    \toprule")
hdr = " & ".join([r"\textbf{" + h + "}" for h in ["Method"] + _TEX_HEADS])
print(f"    {hdr} \\\\")
print(r"    \midrule")

for row in rows:
    name_tex = row["Method"]
    if "(Ours)" in name_tex:
        name_tex = name_tex.replace("(Ours)", r"(\textbf{Ours})")
    name_tex = name_tex.replace("_", r"\_")
    vals = []
    for m in _MCOLS:
        mn, sd = row[f"{m}_mean"], row[f"{m}_std"]
        best_mn = max(r[f"{m}_mean"] for r in rows)
        cell = f"{mn:.3f}" + r"\tiny{$\pm$" + f"{sd:.3f}" + "}"
        if abs(mn - best_mn) < 1e-6:
            cell = r"\textbf{" + cell + "}"
        vals.append(cell)
    print(f"    {name_tex} & {' & '.join(vals)} \\\\")

print(r"    \bottomrule")
print(r"  \end{tabular}")
print(r"\end{table}")